In [ ]:
#Install

!pip install -q evaluate roboflow datasets transformers[torch]

import os, shutil, yaml, torch, numpy as np, evaluate
from roboflow import Roboflow
from datasets import load_dataset
from transformers import (
    ViTImageProcessor,
    ViTForImageClassification,
    TrainingArguments,
    DefaultDataCollator,
    Trainer
)
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt

In [ ]:
#Step 2: Download data

rf = Roboflow(api_key="yRqyBbimhh1vgoeZs2Gx")

projects = [
    ("robotics-lab-1", "soil-moisture-v4", 3),
    ("robotics-lab-1", "soil-moisture-v4-ir", 1),
    ("robotics-lab-1", "soil-moisture-v4-uv", 1),
    ("robotics-lab-1", "soil-moisture-ir", 1),
    ("robotics-lab-1", "soil-moisture-5sagf", 1),
    ("robotics-lab-1", "soil_moisture_september", 8),
    ("robotics-lab-1", "soil_moisture_stir_september", 4)
]

BASE_DIR = '/kaggle/working/source_data'
MASTER_DIR = '/kaggle/working/Master_Soil_Moisture'
os.makedirs(BASE_DIR, exist_ok=True)

for workspace, proj_name, ver in projects:
    try:
        project = rf.workspace(workspace).project(proj_name)
        dataset = project.version(ver).download("yolov5", 
                  location=os.path.join(BASE_DIR, proj_name))
    except Exception as e:
        print(f"Skipping {proj_name}: {e}")

In [ ]:
#Step 3: Check what classes actually exist before mapping
for proj_folder in os.listdir(BASE_DIR):
    yaml_path = os.path.join(BASE_DIR, proj_folder, 'data.yaml')
    if os.path.exists(yaml_path):
        with open(yaml_path, 'r') as f:
            data = yaml.safe_load(f)
        print(f"{proj_folder}: {data['names']}")

In [ ]:
#Step 4: Consolidation and Mapping

if os.path.exists(MASTER_DIR):
    shutil.rmtree(MASTER_DIR)

# Correct mapping based on actual class names
mapping = {
    # Numeric classes - already correct
    '0': '0', '1': '1', '2': '2', '3': '3', '4': '4',
    '5': '5', '6': '6', '7': '7', '8': '8', '9': '9', '10': '10',
    # Named classes from soil-moisture-5sagf and soil-moisture-ir
    'soil-moisture-1.0': '1',
    'soil-moisture-2.0': '2',
    'soil-moisture-3.0': '3',
    'soil-moisture-5.0': '5',
    'soil-moisture-8.2': '8',
    # Level_X format from september datasets (added after Roboflow class rename)
    'Level_0': '0', 'Level_1': '1', 'Level_2': '2', 'Level_3': '3',
    'Level_4': '4', 'Level_5': '5', 'Level_6': '6', 'Level_7': '7',
    'Level_8': '8', 'Level_9': '9', 'Level_10': '10',
}

for proj_folder in os.listdir(BASE_DIR):
    yaml_path = os.path.join(BASE_DIR, proj_folder, 'data.yaml')
    if not os.path.exists(yaml_path):
        continue

    with open(yaml_path, 'r') as f:
        class_names = yaml.safe_load(f)['names']

    for split in ['train', 'valid', 'test']:
        img_src = os.path.join(BASE_DIR, proj_folder, split, 'images')
        lbl_src = os.path.join(BASE_DIR, proj_folder, split, 'labels')
        target_split = 'validation' if split == 'valid' else split

        if not os.path.exists(img_src):
            continue

        for img_file in os.listdir(img_src):
            lbl_file = img_file.rsplit('.', 1)[0] + '.txt'
            lbl_p = os.path.join(lbl_src, lbl_file)

            if not os.path.exists(lbl_p):
                continue

            with open(lbl_p, 'r') as f:
                lines = f.readlines()
            if not lines:
                continue

            raw_name = str(class_names[int(lines[0].split()[0])])
            clean_name = mapping.get(raw_name, None)

            if clean_name is None:
                print(f"Unmapped class: {raw_name} in {proj_folder}")
                continue

            dest = os.path.join(MASTER_DIR, target_split, clean_name)
            os.makedirs(dest, exist_ok=True)
            unique_img = f"{proj_folder}_{img_file}"
            shutil.copy(os.path.join(img_src, img_file),  # ← fixed
                        os.path.join(dest, unique_img))

print("Consolidation complete!")

In [ ]:
#Step 4B: Correcting folder names

# Step 4B: Build HuggingFace class index correction map
import os

MASTER_DIR = '/kaggle/working/Master_Soil_Moisture'

# Build correction map: HuggingFace alphabetical idx -> correct numerical idx
folders = sorted(os.listdir(os.path.join(MASTER_DIR, 'train')))
hf_to_correct = {}
for idx, folder in enumerate(folders):
    hf_to_correct[idx] = int(folder)

print("HuggingFace alphabetical index -> correct numerical class:")
for hf_idx, correct_idx in hf_to_correct.items():
    status = "✓" if hf_idx == correct_idx else "✗ FIXED"
    print(f"  hf_idx {hf_idx} -> class {correct_idx} {status}")

In [ ]:
#Step 5: Verify Consolidation

for split in ['train', 'validation', 'test']:
    split_path = os.path.join(MASTER_DIR, split)
    if os.path.exists(split_path):
        classes = os.listdir(split_path)
        total = sum(len(os.listdir(os.path.join(split_path, c))) for c in classes)
        print(f"\n{split}: {len(classes)} classes, {total} images")
        for c in sorted(classes):
            count = len(os.listdir(os.path.join(split_path, c)))
            print(f"  Class {c}: {count} images")

In [ ]:
#Step 6: Loading Dataset
from datasets import load_dataset, Image as HFImage

raw_ds = load_dataset(
    "imagefolder",
    data_dir=MASTER_DIR,
    drop_labels=False
)

raw_ds = raw_ds.cast_column("image", HFImage(decode=True))

# Remap HuggingFace alphabetical indices to correct numerical indices
# hf_to_correct was built in Step 4B
def remap_label(example):
    example['label'] = hf_to_correct[example['label']]
    return example

raw_ds = raw_ds.map(remap_label)
print("Labels remapped to correct numerical indices")
print(raw_ds)

In [ ]:
#Step 7: Defining Processor

from transformers import ViTImageProcessor

processor = ViTImageProcessor.from_pretrained('google/vit-base-patch16-224-in21k')
print("Processor loaded!")

In [ ]:
#Step 8: Augmentation Transformation (Phase 2 fixed version)

# Step 8 REVISED — Fixed Augmentation
from PIL import Image as PILImage
from torchvision import transforms
import torch

# Augmentation ONLY — no ToTensor or Normalize here
train_augmentation = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(
        brightness=0.3,
        contrast=0.3,
        saturation=0.2,
        hue=0.1
    ),
    transforms.RandomResizedCrop(
        224,
        scale=(0.7, 1.0),
        ratio=(0.8, 1.2)
    ),
    transforms.GaussianBlur(
        kernel_size=3,
        sigma=(0.1, 2.0)
    ),
    transforms.RandomAdjustSharpness(
        sharpness_factor=2,
        p=0.3
    ),
    # NO ToTensor or Normalize — processor handles this
])

def transform_train(example_batch):
    augmented_images = [
        train_augmentation(img.convert("RGB")) 
        for img in example_batch['image']
    ]
    inputs = processor(
        images=augmented_images,
        return_tensors='pt'
    )
    inputs['labels'] = example_batch['label']
    return inputs

def transform_val(example_batch):
    inputs = processor(
        images=[img.convert("RGB") for img in example_batch['image']],
        return_tensors='pt'
    )
    inputs['labels'] = example_batch['label']
    return inputs

# Apply transforms
prepared_ds_train = raw_ds['train'].with_transform(transform_train)
prepared_ds_val   = raw_ds['validation'].with_transform(transform_val)
prepared_ds_test  = raw_ds['test'].with_transform(transform_val)

print("Augmentation pipeline ready!")

In [ ]:
#Step 9: Full ViT Training, whole image Phase 1 and 2

# Step 9 REVISED — Phase 2: Training with Augmentation
import evaluate
import numpy as np
from transformers import (
    ViTForImageClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback
)

# Model
model = ViTForImageClassification.from_pretrained(
    'google/vit-base-patch16-224-in21k',
    num_labels=11,
    id2label={i: f"Level {i}" for i in range(11)},
    label2id={f"Level {i}": i for i in range(11)},
    ignore_mismatched_sizes=True,
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1
)

# Metric
metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return metric.compute(predictions=predictions, references=labels)

# Training Arguments — Phase 2 UPDATED
training_args = TrainingArguments(
    output_dir="./results_v3",
    save_total_limit=1,
    save_strategy="no",              # no checkpoints during training
    load_best_model_at_end=False,    # must be False when save_strategy="no"
    eval_strategy="epoch",
    logging_steps=5,
    num_train_epochs=25,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=100,
    lr_scheduler_type="cosine",
    metric_for_best_model="accuracy",
    greater_is_better=True,
    remove_unused_columns=False,
    label_smoothing_factor=0.1,
)

# Trainer — no EarlyStoppingCallback since save_strategy="no"
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=prepared_ds_train,
    eval_dataset=prepared_ds_val,
    processing_class=processor,
    compute_metrics=compute_metrics,
)

trainer.train()

# Save once after training completes
trainer.save_model('./results_v3/final_model')
processor.save_pretrained('./results_v3/final_model')
print("Model saved!")

In [ ]:
#Step 9B: Save only once after training ends
trainer.save_model('./results_v3/final_model')
processor.save_pretrained('./results_v3/final_model')
print("Model saved!")

In [ ]:
#Step 10: Metrics Phase 1 and 2 (Loss Curve, Classification Table, Accuracy Table, and Confusion Matrix)

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (classification_report, 
                             confusion_matrix, 
                             ConfusionMatrixDisplay)

# Exact values from your training output
# Replace these in Step 10 with Phase 2 actual values

# Step 10 REVISED — Phase 2 Updated Values
train_losses = [4.673071, 4.453415, 4.095893, 3.647770, 3.428395,
                3.116595, 2.688093, 2.507006, 2.278150, 2.079329,
                1.984986, 1.814550, 1.846718, 1.590630, 1.656840,
                1.560463, 1.605761, 1.555979, 1.414505, 1.446769,
                1.404795, 1.368590, 1.344135, 1.435816, 1.418155]

val_losses = [4.751688, 4.460595, 4.137558, 3.761130, 3.428313,
              3.167595, 2.809613, 2.599695, 2.414394, 2.290302,
              2.129147, 1.914390, 1.913347, 1.790328, 1.751759,
              1.676690, 1.670827, 1.611479, 1.599333, 1.581134,
              1.574428, 1.565577, 1.563511, 1.564832, 1.564560]

val_accuracies = [0.133005, 0.339901, 0.389163, 0.453202, 0.556650,
                  0.748768, 0.837438, 0.857143, 0.876847, 0.862069,
                  0.866995, 0.940887, 0.886700, 0.896552, 0.921182,
                  0.945813, 0.926108, 0.945813, 0.945813, 0.945813,
                  0.945813, 0.945813, 0.945813, 0.945813, 0.945813]

epochs = range(1, 26)

# 1. Classification Report
print("\n=== CLASSIFICATION REPORT ===")
predictions = trainer.predict(prepared_ds_test)
y_pred = np.argmax(predictions.predictions, axis=1)
y_true = predictions.label_ids
print(classification_report(y_true, y_pred, target_names=class_names))

# 2. Accuracy Graph with target line
plt.figure(figsize=(10, 5))
plt.plot(epochs, val_accuracies, label='Validation Accuracy', marker='o', color='blue')
plt.axhline(y=0.98, color='r', linestyle='--', label='Target (98%)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Validation Accuracy vs Target')
plt.ylim([0.8, 1.0])
plt.legend()
plt.grid(True)
plt.savefig('accuracy_graph_phase2.png', dpi=150, bbox_inches='tight')
plt.show()
print("Accuracy graph saved!")

# 3. Loss Curve
plt.figure(figsize=(10, 5))
plt.plot(epochs, train_losses, label='Training Loss', marker='o', color='blue')
plt.plot(epochs, val_losses, label='Validation Loss', marker='s', color='orange')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss Curve')
plt.legend()
plt.grid(True)
plt.savefig('loss_curve_phase2.png', dpi=150, bbox_inches='tight')
plt.show()
print("Loss curve saved!")

# 4. Confusion Matrix - with full label names
class_names_full = [f"Soil Moisture Level {i}" for i in range(11)]

plt.figure(figsize=(14, 12))
cm = confusion_matrix(y_true, y_pred)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names_full)
disp.plot(cmap='Blues', xticks_rotation=45)
plt.title('Confusion Matrix — Soil Moisture ViT Classifier')
plt.tight_layout()
plt.savefig('confusion_matrix_phase2.png', dpi=150, bbox_inches='tight')
plt.show()
print("Confusion matrix saved!")

In [ ]:
#Images Generation and Inference Test

# Step 11: Image Generation — FIXED
import os
import random
import zipfile
import numpy as np
import torch
import yaml
from PIL import Image, ImageDraw
from transformers import ViTForImageClassification, ViTImageProcessor

# Setup
SOURCE_DIR = '/kaggle/working/source_data'
OUTPUT_DIR = '/kaggle/working/inference_results'
ZIP_PATH = '/kaggle/working/soil_moisture_inference_50.zip'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ── DEVICE FIX ──────────────────────────────────────────────────────────────
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)
model.eval()
# ────────────────────────────────────────────────────────────────────────────

# Class mapping
mapping = {
    'soil-moisture-1.0': 'Level 1', 'soil-moisture-2.0': 'Level 2',
    'soil-moisture-3.0': 'Level 3', 'soil-moisture-5.0': 'Level 5',
    'soil-moisture-8.2': 'Level 8',
    '0': 'Level 0', '1': 'Level 1', '2': 'Level 2', '3': 'Level 3',
    '4': 'Level 4', '5': 'Level 5', '6': 'Level 6', '7': 'Level 7',
    '8': 'Level 8', '9': 'Level 9', '10': 'Level 10',
}

# Samples per dataset
samples_per_dataset = {
    'soil-moisture-v4':             8,
    'soil-moisture-v4-ir':          7,
    'soil-moisture-v4-uv':          7,
    'soil-moisture-ir':             7,
    'soil-moisture-5sagf':          7,
    'soil_moisture_september':      7,
    'soil_moisture_stir_september': 7,
}

def annotate_image(img, dataset_name, img_id, pred_label, true_label):
    img = img.convert("RGB").resize((640, 680))
    draw = ImageDraw.Draw(img)
    draw.rectangle([0, 0, 640, 160], fill=(0, 0, 0))
    draw.text((10, 8),  f"Dataset: {dataset_name}", fill=(255, 255, 255))
    img_id_display = img_id[:50] + '...' if len(img_id) > 50 else img_id
    draw.text((10, 35), f"Image ID: {img_id_display}", fill=(255, 255, 255))
    pred_color = (0, 255, 0) if pred_label == true_label else (255, 0, 0)
    draw.text((10, 62), f"Predicted:    {pred_label}", fill=pred_color)
    draw.text((10, 89), f"Ground Truth: {true_label}", fill=(255, 255, 0))
    result_text  = "CORRECT" if pred_label == true_label else "INCORRECT"
    result_color = (0, 255, 0) if pred_label == true_label else (255, 0, 0)
    draw.rectangle([0, 640, 640, 680], fill=(0, 0, 0))
    draw.text((10, 648), result_text, fill=result_color)
    return img

# Process each dataset
all_saved = []
sample_counter = 1

for dataset_name, count in samples_per_dataset.items():
    dataset_path = os.path.join(SOURCE_DIR, dataset_name)
    if not os.path.exists(dataset_path):
        print(f"Skipping {dataset_name} — folder not found")
        continue

    img_dir = os.path.join(dataset_path, 'test', 'images')
    lbl_dir = os.path.join(dataset_path, 'test', 'labels')
    if not os.path.exists(img_dir):
        img_dir = os.path.join(dataset_path, 'valid', 'images')
        lbl_dir = os.path.join(dataset_path, 'valid', 'labels')
    if not os.path.exists(img_dir):
        print(f"No images found for {dataset_name}")
        continue

    yaml_path = os.path.join(dataset_path, 'data.yaml')
    with open(yaml_path, 'r') as f:
        class_names = yaml.safe_load(f)['names']

    all_imgs = [f for f in os.listdir(img_dir)
                if f.endswith(('.jpg', '.jpeg', '.png'))]
    selected = random.sample(all_imgs, min(count, len(all_imgs)))

    for img_file in selected:
        img_path = os.path.join(img_dir, img_file)
        lbl_path = os.path.join(lbl_dir, img_file.rsplit('.', 1)[0] + '.txt')

        true_label = 'Unknown'
        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as f:
                lines = f.readlines()
            if lines:
                class_id  = int(lines[0].split()[0])
                raw_name  = str(class_names[class_id])
                true_label = mapping.get(raw_name, raw_name)

        # Run inference
        img    = Image.open(img_path).convert("RGB")
        inputs = processor(images=img, return_tensors="pt")
        inputs = {k: v.to(device) for k, v in inputs.items()}  # ← FIX

        with torch.no_grad():
            outputs = model(**inputs)

        pred_id    = outputs.logits.argmax(-1).item()
        pred_label = f"Level {pred_id}"

        img_id   = img_file.rsplit('.', 1)[0]
        annotated = annotate_image(img, dataset_name, img_id, pred_label, true_label)
        save_name = f"sample_{sample_counter:02d}_{dataset_name}.jpg"
        save_path = os.path.join(OUTPUT_DIR, save_name)
        annotated.save(save_path)
        all_saved.append(save_path)
        print(f"Sample {sample_counter:02d} | {dataset_name} | {img_id} | "
              f"Pred: {pred_label} | Truth: {true_label}")
        sample_counter += 1

# Zip all images
with zipfile.ZipFile(ZIP_PATH, 'w') as zipf:
    for file_path in all_saved:
        zipf.write(file_path, os.path.basename(file_path))

print(f"\n{len(all_saved)} images saved and zipped!")
print(f"ZIP location: {ZIP_PATH}")

from IPython.display import FileLink
display(FileLink(ZIP_PATH))

In [ ]:
#Step 11b: Check for bounding
import os
import yaml

# Check bounding box labels in each dataset
SOURCE_DIR = '/kaggle/working/source_data'

for proj_folder in os.listdir(SOURCE_DIR):
    proj_path = os.path.join(SOURCE_DIR, proj_folder)
    
    # Check yaml for class names
    yaml_path = os.path.join(proj_path, 'data.yaml')
    if not os.path.exists(yaml_path):
        continue
    with open(yaml_path, 'r') as f:
        class_names = yaml.safe_load(f)['names']
    
    # Sample one label file
    for split in ['train', 'valid', 'test']:
        lbl_dir = os.path.join(proj_path, split, 'labels')
        if not os.path.exists(lbl_dir):
            continue
        label_files = os.listdir(lbl_dir)
        if label_files:
            sample_lbl = os.path.join(lbl_dir, label_files[0])
            with open(sample_lbl, 'r') as f:
                content = f.read().strip()
            print(f"\n{proj_folder} | {split}")
            print(f"Classes: {class_names}")
            print(f"Sample label: {content[:200]}")
            break


In [ ]:
# Step 12: Crop Laser Regions from All Datasets
import os
import shutil
import yaml
from PIL import Image

SOURCE_DIR = '/kaggle/working/source_data'
LASER_DIR  = '/kaggle/working/Master_Laser_Crops'

# Mapping consistent with previous steps
mapping = {
    'soil-moisture-1.0': '1', 'soil-moisture-2.0': '2',
    'soil-moisture-3.0': '3', 'soil-moisture-5.0': '5',
    'soil-moisture-8.2': '8',
    '0': '0', '1': '1', '2': '2', '3': '3', '4': '4',
    '5': '5', '6': '6', '7': '7', '8': '8', '9': '9', '10': '10',
    # Level_X format from september datasets
    'Level_0': '0', 'Level_1': '1', 'Level_2': '2', 'Level_3': '3',
    'Level_4': '4', 'Level_5': '5', 'Level_6': '6', 'Level_7': '7',
    'Level_8': '8', 'Level_9': '9', 'Level_10': '10',
}

def crop_laser(img, x_center, y_center, width, height, padding=0.05):
    """Crop laser region from image using YOLO normalized coordinates."""
    W, H = img.size
    
    # Convert normalized coords to pixel coords
    x1 = int((x_center - width/2  - padding) * W)
    y1 = int((y_center - height/2 - padding) * H)
    x2 = int((x_center + width/2  + padding) * W)
    y2 = int((y_center + height/2 + padding) * H)
    
    # Clamp to image boundaries
    x1 = max(0, x1)
    y1 = max(0, y1)
    x2 = min(W, x2)
    y2 = min(H, y2)
    
    # If bounding box is full image, return as-is
    if width >= 0.95 and height >= 0.95:
        return img
    
    return img.crop((x1, y1, x2, y2))

if os.path.exists(LASER_DIR):
    shutil.rmtree(LASER_DIR)

skipped = 0
copied  = 0

for proj_folder in os.listdir(SOURCE_DIR):
    proj_path = os.path.join(SOURCE_DIR, proj_folder)
    yaml_path = os.path.join(proj_path, 'data.yaml')
    if not os.path.exists(yaml_path):
        continue

    with open(yaml_path, 'r') as f:
        class_names = yaml.safe_load(f)['names']

    for split in ['train', 'valid', 'test']:
        img_dir = os.path.join(proj_path, split, 'images')
        lbl_dir = os.path.join(proj_path, split, 'labels')
        target_split = 'validation' if split == 'valid' else split

        if not os.path.exists(img_dir):
            continue

        for img_file in os.listdir(img_dir):
            if not img_file.endswith(('.jpg', '.jpeg', '.png')):
                continue

            lbl_file = img_file.rsplit('.', 1)[0] + '.txt'
            lbl_path = os.path.join(lbl_dir, lbl_file)

            if not os.path.exists(lbl_path):
                skipped += 1
                continue

            with open(lbl_path, 'r') as f:
                lines = f.readlines()
            if not lines:
                skipped += 1
                continue

            # Parse label
            parts    = lines[0].strip().split()
            class_id = int(parts[0])
            x_center = float(parts[1])
            y_center = float(parts[2])
            width    = float(parts[3])
            height   = float(parts[4])

            raw_name   = str(class_names[class_id])
            clean_name = mapping.get(raw_name, None)
            if clean_name is None:
                skipped += 1
                continue

            # Crop and save
            img_path = os.path.join(img_dir, img_file)
            img      = Image.open(img_path).convert("RGB")
            cropped  = crop_laser(img, x_center, y_center, width, height)

            dest = os.path.join(LASER_DIR, target_split, clean_name)
            os.makedirs(dest, exist_ok=True)
            unique_name = f"{proj_folder}_{img_file}"
            cropped.save(os.path.join(dest, unique_name))
            copied += 1

print(f"Laser crops complete! {copied} saved, {skipped} skipped")
print(f"Saved to: {LASER_DIR}")


In [ ]:
# Step 13: Verify Laser Crop Dataset
for split in ['train', 'validation', 'test']:
    split_path = os.path.join(LASER_DIR, split)
    if os.path.exists(split_path):
        classes = os.listdir(split_path)
        total   = sum(len(os.listdir(os.path.join(split_path, c))) 
                      for c in classes)
        print(f"\n{split}: {len(classes)} classes, {total} images")
        for c in sorted(classes):
            count = len(os.listdir(os.path.join(split_path, c)))
            print(f"  Class {c}: {count} images")

# Visualize sample crops
import matplotlib.pyplot as plt
import random

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

train_path = os.path.join(LASER_DIR, 'train')
all_classes = sorted(os.listdir(train_path))

for i, cls in enumerate(all_classes[:10]):
    cls_path = os.path.join(train_path, cls)
    sample   = random.choice(os.listdir(cls_path))
    img      = Image.open(os.path.join(cls_path, sample))
    axes[i].imshow(img)
    axes[i].set_title(f"Level {cls}", fontsize=12)
    axes[i].axis('off')

plt.suptitle('Sample Laser Crops by Moisture Level', fontsize=14)
plt.tight_layout()
plt.savefig('laser_crops_sample.png', dpi=150, bbox_inches='tight')
plt.show()
print("Sample crops visualized!")


In [ ]:
# Step 14: Load Laser Crop Dataset and Train ViT
from datasets import load_dataset
from datasets import Image as HFImage

# Load cropped laser dataset
laser_ds = load_dataset(
    "imagefolder",
    data_dir=LASER_DIR,
    drop_labels=False
)
laser_ds = laser_ds.cast_column("image", HFImage(decode=True))
print(laser_ds)

# Augmentation for laser crops
from torchvision import transforms
import torch

train_augmentation = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=20),
    transforms.ColorJitter(
        brightness=0.4,   # more aggressive — laser intensity varies widely
        contrast=0.4,
        saturation=0.3,
        hue=0.1
    ),
    transforms.RandomResizedCrop(
        224,
        scale=(0.8, 1.0)  # less aggressive crop — laser region is already small
    ),
    transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
])

def transform_train(example_batch):
    augmented = [
        train_augmentation(img.convert("RGB"))
        for img in example_batch['image']
    ]
    inputs = processor(images=augmented, return_tensors='pt')
    inputs['labels'] = example_batch['label']
    return inputs

def transform_val(example_batch):
    inputs = processor(
        images=[img.convert("RGB") for img in example_batch['image']],
        return_tensors='pt'
    )
    inputs['labels'] = example_batch['label']
    return inputs

laser_train = laser_ds['train'].with_transform(transform_train)
laser_val   = laser_ds['validation'].with_transform(transform_val)
laser_test  = laser_ds['test'].with_transform(transform_val)

print("Laser dataset ready!")


In [ ]:
# Step 15: Phase 3 Train ViT on Laser Crops(40 epochs, no augmentation)
import evaluate
import numpy as np
from transformers import (
    ViTForImageClassification,
    TrainingArguments,
    Trainer
)

# Fresh model
model_v3 = ViTForImageClassification.from_pretrained(
    'google/vit-base-patch16-224-in21k',
    num_labels=11,
    id2label={i: f"Level {i}" for i in range(11)},
    label2id={f"Level {i}": i for i in range(11)},
    ignore_mismatched_sizes=True,
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1
)

metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return metric.compute(predictions=predictions, references=labels)

training_args_v3 = TrainingArguments(
    output_dir="./results_v4",
    save_strategy="no",
    eval_strategy="epoch",
    logging_steps=5,
    num_train_epochs=40,
    per_device_train_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=100,
    lr_scheduler_type="cosine",
    load_best_model_at_end=False,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    remove_unused_columns=False,
    label_smoothing_factor=0.1,
    save_total_limit=1,
)

trainer_v3 = Trainer(
    model=model_v3,
    args=training_args_v3,
    train_dataset=laser_train,
    eval_dataset=laser_val,
    processing_class=processor,
    compute_metrics=compute_metrics,
)

trainer_v3.train()

# Save final model
trainer_v3.save_model('./results_v4/final_model')
processor.save_pretrained('./results_v4/final_model')
print("Phase 3 model saved!")

In [ ]:
# Step 15B — Phase 3 Extended Metrics
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (classification_report,
                             confusion_matrix,
                             ConfusionMatrixDisplay)

train_losses_p3 = [4.775843, 4.678317, 4.532147, 4.234828, 4.019403,
                   3.805770, 3.525551, 3.378486, 3.212287, 3.031305,
                   2.988930, 2.706341, 2.556252, 2.478303, 2.368372,
                   2.314680, 2.132662, 2.146253, 2.074467, 2.028231,
                   2.048648, 1.891575, 1.909996, 1.825216, 1.774547,
                   1.766110, 1.808014, 1.757672, 1.694626, 1.817509,
                   1.637901, 1.668483, 1.637347, 1.635563, 1.726993,
                   1.575352, 1.651339, 1.665032, 1.657816, 1.623149]

val_losses_p3 = [4.758066, 4.661328, 4.541890, 4.323533, 4.126867,
                 3.939586, 3.743506, 3.528446, 3.339040, 3.205756,
                 3.013563, 2.824293, 2.670386, 2.564347, 2.484676,
                 2.474155, 2.350775, 2.291309, 2.195544, 2.233965,
                 2.148098, 2.096861, 2.102739, 2.065074, 2.061624,
                 2.010844, 1.997552, 1.987275, 1.987496, 1.966841,
                 1.933921, 1.922749, 1.919895, 1.929911, 1.922580,
                 1.921657, 1.912679, 1.914333, 1.914521, 1.914572]

val_accuracies_p3 = [0.167488, 0.236453, 0.275862, 0.266010, 0.399015,
                     0.472906, 0.561576, 0.605911, 0.689655, 0.709360,
                     0.768473, 0.812808, 0.827586, 0.817734, 0.822660,
                     0.837438, 0.822660, 0.822660, 0.837438, 0.832512,
                     0.852217, 0.847291, 0.837438, 0.842365, 0.857143,
                     0.852217, 0.852217, 0.837438, 0.852217, 0.852217,
                     0.862069, 0.876847, 0.881773, 0.862069, 0.876847,
                     0.871921, 0.876847, 0.876847, 0.876847, 0.876847]

epochs_p3 = range(1, 41)
class_names = [f"Level {i}" for i in range(11)]

# Classification Report
print("\n=== PHASE 3 EXTENDED CLASSIFICATION REPORT ===")
predictions_p3 = trainer_v3.predict(laser_test)
y_pred_p3 = np.argmax(predictions_p3.predictions, axis=1)
y_true_p3 = predictions_p3.label_ids
print(classification_report(y_true_p3, y_pred_p3, target_names=class_names))

# Accuracy Graph
plt.figure(figsize=(10, 5))
plt.plot(epochs_p3, val_accuracies_p3, label='Validation Accuracy',
         marker='o', color='blue')
plt.axhline(y=0.98, color='r', linestyle='--', label='Target (98%)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Phase 3 Extended — Validation Accuracy vs Target')
plt.ylim([0.1, 1.0])
plt.legend()
plt.grid(True)
plt.savefig('accuracy_graph_phase3_extended.png', dpi=150, bbox_inches='tight')
plt.show()

# Loss Curve
plt.figure(figsize=(10, 5))
plt.plot(epochs_p3, train_losses_p3, label='Training Loss',
         marker='o', color='blue')
plt.plot(epochs_p3, val_losses_p3, label='Validation Loss',
         marker='s', color='orange')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Phase 3 Extended — Training and Validation Loss Curve')
plt.legend()
plt.grid(True)
plt.savefig('loss_curve_phase3_extended.png', dpi=150, bbox_inches='tight')
plt.show()

# Confusion Matrix
class_names_full = [f"Soil Moisture Level {i}" for i in range(11)]
plt.figure(figsize=(14, 12))
cm_p3 = confusion_matrix(y_true_p3, y_pred_p3)
disp  = ConfusionMatrixDisplay(confusion_matrix=cm_p3,
                                display_labels=class_names_full)
disp.plot(cmap='Blues', xticks_rotation=45)
plt.title('Phase 3 Extended Confusion Matrix — Laser Crop ViT Classifier')
plt.tight_layout()
plt.savefig('confusion_matrix_phase3_extended.png', dpi=150, bbox_inches='tight')
plt.show()
print("Phase 3 extended metrics saved!")

In [ ]:
# Step 16 — Phase 4A: Generate Noisy Augmented Images
import os
import numpy as np
from PIL import Image
import random

LASER_DIR = '/kaggle/working/Master_Laser_Crops'
train_path = os.path.join(LASER_DIR, 'train')

def add_gaussian_noise(img, mean=0, std=25):
    """Add Gaussian noise to image."""
    img_array = np.array(img).astype(np.float32)
    noise = np.random.normal(mean, std, img_array.shape)
    noisy = np.clip(img_array + noise, 0, 255).astype(np.uint8)
    return Image.fromarray(noisy)

def add_salt_pepper_noise(img, amount=0.05):
    """Add salt and pepper noise to image."""
    img_array = np.array(img).astype(np.uint8)
    noisy = img_array.copy()
    
    # Salt — white pixels
    num_salt = int(amount * img_array.size * 0.5)
    salt_coords = [np.random.randint(0, i, num_salt) 
                   for i in img_array.shape[:2]]
    noisy[salt_coords[0], salt_coords[1]] = 255
    
    # Pepper — black pixels
    num_pepper = int(amount * img_array.size * 0.5)
    pepper_coords = [np.random.randint(0, i, num_pepper) 
                     for i in img_array.shape[:2]]
    noisy[pepper_coords[0], pepper_coords[1]] = 0
    
    return Image.fromarray(noisy)

def flip_image(img):
    """Apply random horizontal or vertical flip."""
    choice = random.randint(0, 2)
    if choice == 0:
        return img.transpose(Image.FLIP_LEFT_RIGHT)
    elif choice == 1:
        return img.transpose(Image.FLIP_TOP_BOTTOM)
    else:
        # Both flips
        img = img.transpose(Image.FLIP_LEFT_RIGHT)
        return img.transpose(Image.FLIP_TOP_BOTTOM)

# Generate augmented copies
augmented_count = 0

for class_folder in os.listdir(train_path):
    class_path = os.path.join(train_path, class_folder)
    if not os.path.isdir(class_path):
        continue

    original_files = [f for f in os.listdir(class_path)
                      if f.endswith(('.jpg', '.jpeg', '.png'))]

    for img_file in original_files:
        img_path = os.path.join(class_path, img_file)
        img = Image.open(img_path).convert("RGB")
        base_name = img_file.rsplit('.', 1)[0]

        # Copy 1 — flip + Gaussian noise
        aug1 = flip_image(img)
        aug1 = add_gaussian_noise(aug1, mean=0, std=25)
        aug1_path = os.path.join(class_path, f"{base_name}_aug_gaussian.jpg")
        aug1.save(aug1_path)

        # Copy 2 — flip + Salt & Pepper noise
        aug2 = flip_image(img)
        aug2 = add_salt_pepper_noise(aug2, amount=0.05)
        aug2_path = os.path.join(class_path, f"{base_name}_aug_saltpepper.jpg")
        aug2.save(aug2_path)

        augmented_count += 2

print(f"Augmentation complete!")
print(f"Original images: 717")
print(f"New augmented images added: {augmented_count}")
print(f"Total training images: {717 + augmented_count}")

# Verify counts per class
print("\nPer class breakdown:")
for class_folder in sorted(os.listdir(train_path)):
    class_path = os.path.join(train_path, class_folder)
    if os.path.isdir(class_path):
        count = len(os.listdir(class_path))
        print(f"  Class {class_folder}: {count} images")

In [ ]:
# Step 17 — Reload Expanded Dataset
from datasets import load_dataset
from datasets import Image as HFImage

# Reload laser dataset — now includes augmented images
laser_ds_aug = load_dataset(
    "imagefolder",
    data_dir=LASER_DIR,
    drop_labels=False
)
laser_ds_aug = laser_ds_aug.cast_column("image", HFImage(decode=True))
print(laser_ds_aug)
print(f"Training set expanded to: {laser_ds_aug['train'].num_rows} images")

In [ ]:
# Step 18 — Apply Transforms to Expanded Dataset
from torchvision import transforms
import torch

# Lighter augmentation now since noise already added physically
train_augmentation_v2 = transforms.Compose([
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=15),
    transforms.ColorJitter(
        brightness=0.3,
        contrast=0.3,
        saturation=0.2,
        hue=0.1
    ),
    transforms.RandomResizedCrop(
        224,
        scale=(0.8, 1.0)
    ),
])

def transform_train_aug(example_batch):
    augmented = [
        train_augmentation_v2(img.convert("RGB"))
        for img in example_batch['image']
    ]
    inputs = processor(images=augmented, return_tensors='pt')
    inputs['labels'] = example_batch['label']
    return inputs

def transform_val(example_batch):
    inputs = processor(
        images=[img.convert("RGB") for img in example_batch['image']],
        return_tensors='pt'
    )
    inputs['labels'] = example_batch['label']
    return inputs

laser_train_aug = laser_ds_aug['train'].with_transform(transform_train_aug)
laser_val_aug   = laser_ds_aug['validation'].with_transform(transform_val)
laser_test_aug  = laser_ds_aug['test'].with_transform(transform_val)

print("Expanded dataset transforms ready!")
print(f"Training: {laser_ds_aug['train'].num_rows} images")
print(f"Validation: {laser_ds_aug['validation'].num_rows} images")
print(f"Test: {laser_ds_aug['test'].num_rows} images")

In [ ]:
# Step 19 — Train ViT on Augmented Laser Crops
import evaluate
import numpy as np
from transformers import (
    ViTForImageClassification,
    TrainingArguments,
    Trainer
)

# Fresh model
model_v4 = ViTForImageClassification.from_pretrained(
    'google/vit-base-patch16-224-in21k',
    num_labels=11,
    id2label={i: f"Level {i}" for i in range(11)},
    label2id={f"Level {i}": i for i in range(11)},
    ignore_mismatched_sizes=True,
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1
)

metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return metric.compute(predictions=predictions, references=labels)

training_args_v4 = TrainingArguments(
    output_dir="./results_v5",
    save_strategy="no",
    eval_strategy="epoch",
    logging_steps=5,
    num_train_epochs=40,
    per_device_train_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=100,
    lr_scheduler_type="cosine",
    load_best_model_at_end=False,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    remove_unused_columns=False,
    label_smoothing_factor=0.1,
    save_total_limit=1,
)

trainer_v4 = Trainer(
    model=model_v4,
    args=training_args_v4,
    train_dataset=laser_train_aug,
    eval_dataset=laser_val_aug,
    processing_class=processor,
    compute_metrics=compute_metrics,
)

trainer_v4.train()

# Save final model
trainer_v4.save_model('./results_v5/final_model')
processor.save_pretrained('./results_v5/final_model')
print("Phase 4A model saved!")

In [ ]:
# Step 19B: Phase 4A with Class Weighted Loss
import torch
import numpy as np
import evaluate
from transformers import ViTForImageClassification, TrainingArguments, Trainer

# Compute class weights from training data
class_counts = [34, 59, 78, 65, 109, 105, 42, 53, 65, 36, 71]  # from Step 13 output
total = sum(class_counts)
num_classes = len(class_counts)

# Inverse frequency weighting — rare classes get higher weight
class_weights = torch.tensor(
    [total / (num_classes * count) for count in class_counts],
    dtype=torch.float32
).to('cuda')

print("Class weights:")
for i, w in enumerate(class_weights):
    print(f"  Level {i}: {w:.4f}")

# Custom Trainer with weighted loss
class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss_fn = torch.nn.CrossEntropyLoss(weight=class_weights)
        loss = loss_fn(logits, labels)
        return (loss, outputs) if return_outputs else loss

# Fresh model
model_v4b = ViTForImageClassification.from_pretrained(
    'google/vit-base-patch16-224-in21k',
    num_labels=11,
    id2label={i: f"Level {i}" for i in range(11)},
    label2id={f"Level {i}": i for i in range(11)},
    ignore_mismatched_sizes=True,
    hidden_dropout_prob=0.1,
    attention_probs_dropout_prob=0.1
)

metric = evaluate.load("accuracy")
def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    return metric.compute(predictions=predictions, references=labels)

training_args_v4b = TrainingArguments(
    output_dir="./results_v6",
    save_strategy="no",
    eval_strategy="epoch",
    logging_steps=5,
    num_train_epochs=40,
    per_device_train_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_steps=100,
    lr_scheduler_type="cosine",
    load_best_model_at_end=False,
    metric_for_best_model="accuracy",
    greater_is_better=True,
    remove_unused_columns=False,
    label_smoothing_factor=0.1,
    save_total_limit=1,
)

trainer_v4b = WeightedTrainer(
    model=model_v4b,
    args=training_args_v4b,
    train_dataset=laser_train_aug,
    eval_dataset=laser_val_aug,
    processing_class=processor,
    compute_metrics=compute_metrics,
)

trainer_v4b.train()

# Save
trainer_v4b.save_model('./results_v6/final_model')
processor.save_pretrained('./results_v6/final_model')
print("Phase 4B weighted loss model saved!")

In [ ]:
# Step 20: Phase 4A Metrics

# Fix — import missing and replot confusion matrix only
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

class_names_full = [f"Soil Moisture Level {i}" for i in range(11)]
plt.figure(figsize=(14, 12))
cm_p4a = confusion_matrix(y_true_p4a, y_pred_p4a)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_p4a,
                               display_labels=class_names_full)
disp.plot(cmap='Blues', xticks_rotation=45)
plt.title('Phase 4A Confusion Matrix — Augmented Laser Crop ViT Classifier')
plt.tight_layout()
plt.savefig('confusion_matrix_phase4a.png', dpi=150, bbox_inches='tight')
plt.show()
print("Confusion matrix saved!")


train_losses_p4a = [4.598878, 4.018516, 3.466031, 3.105857, 2.701540,
                    2.416365, 2.183443, 1.930872, 1.865558, 1.660936,
                    1.601121, 1.554153, 1.487123, 1.435534, 1.452652,
                    1.371130, 1.396241, 1.277487, 1.367294, 1.145833,
                    1.172555, 1.357800, 1.288642, 1.175231, 1.273762,
                    1.203880, 1.224792, 1.190668, 1.156583, 1.123374,
                    1.139964, 1.203056, 1.173595, 1.167897, 1.171117,
                    1.172913, 1.196974, 1.106842, 1.134756, 1.133870]

val_losses_p4a = [4.553881, 3.976180, 3.464017, 2.997657, 2.627634,
                  2.384494, 2.262010, 2.055072, 1.935329, 1.903518,
                  1.802516, 1.753705, 1.762875, 1.825698, 1.669055,
                  1.766719, 1.687311, 1.614212, 1.633559, 1.572980,
                  1.654490, 1.615139, 1.625992, 1.633548, 1.589483,
                  1.614638, 1.604377, 1.591931, 1.569810, 1.570749,
                  1.570167, 1.573008, 1.596105, 1.591208, 1.593277,
                  1.589233, 1.593882, 1.597016, 1.597456, 1.597225]

val_accuracies_p4a = [0.216749, 0.344828, 0.566502, 0.778325, 0.812808,
                      0.812808, 0.778325, 0.842365, 0.842365, 0.822660,
                      0.842365, 0.852217, 0.857143, 0.837438, 0.876847,
                      0.847291, 0.871921, 0.886700, 0.876847, 0.896552,
                      0.871921, 0.876847, 0.891626, 0.876847, 0.886700,
                      0.876847, 0.881773, 0.891626, 0.896552, 0.891626,
                      0.881773, 0.891626, 0.886700, 0.881773, 0.881773,
                      0.886700, 0.886700, 0.881773, 0.886700, 0.886700]

epochs_p4a = range(1, 41)
class_names = [f"Level {i}" for i in range(11)]

# Classification Report
print("\n=== PHASE 4A CLASSIFICATION REPORT ===")
predictions_p4a = trainer_v4.predict(laser_test_aug)
y_pred_p4a = np.argmax(predictions_p4a.predictions, axis=1)
y_true_p4a = predictions_p4a.label_ids
print(classification_report(y_true_p4a, y_pred_p4a, target_names=class_names))

# Accuracy Graph
plt.figure(figsize=(10, 5))
plt.plot(epochs_p4a, val_accuracies_p4a, label='Validation Accuracy',
         marker='o', color='blue')
plt.axhline(y=0.98, color='r', linestyle='--', label='Target (98%)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Phase 4A — Validation Accuracy vs Target')
plt.ylim([0.1, 1.0])
plt.legend()
plt.grid(True)
plt.savefig('accuracy_graph_phase4a.png', dpi=150, bbox_inches='tight')
plt.show()

# Loss Curve
plt.figure(figsize=(10, 5))
plt.plot(epochs_p4a, train_losses_p4a, label='Training Loss',
         marker='o', color='blue')
plt.plot(epochs_p4a, val_losses_p4a, label='Validation Loss',
         marker='s', color='orange')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Phase 4A — Training and Validation Loss Curve')
plt.legend()
plt.grid(True)
plt.savefig('loss_curve_phase4a.png', dpi=150, bbox_inches='tight')
plt.show()

# Confusion Matrix
class_names_full = [f"Soil Moisture Level {i}" for i in range(11)]
plt.figure(figsize=(14, 12))
cm_p4a = confusion_matrix(y_true_p4a, y_pred_p4a)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_p4a,
                               display_labels=class_names_full)
disp.plot(cmap='Blues', xticks_rotation=45)
plt.title('Phase 4A Confusion Matrix — Augmented Laser Crop ViT Classifier')
plt.tight_layout()
plt.savefig('confusion_matrix_phase4a.png', dpi=150, bbox_inches='tight')
plt.show()
print("Phase 4A metrics saved!")

In [ ]:
# Step 20B: Phase 4B Metrics
import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import (classification_report,
                             confusion_matrix,
                             ConfusionMatrixDisplay)

train_losses_p4b = [2.293143, 1.928957, 1.661045, 1.403453, 1.121677,
                    1.010281, 0.819675, 0.739883, 0.671993, 0.521686,
                    0.500440, 0.470915, 0.441232, 0.418277, 0.377515,
                    0.331545, 0.337027, 0.261187, 0.324425, 0.231847,
                    0.224760, 0.282517, 0.225149, 0.176197, 0.212028,
                    0.188740, 0.206023, 0.179401, 0.159163, 0.140444,
                    0.139795, 0.232164, 0.192412, 0.164299, 0.171826,
                    0.165090, 0.184149, 0.133453, 0.153742, 0.133313]

val_losses_p4b = [2.288795, 2.021616, 1.686626, 1.453947, 1.222767,
                  1.093630, 0.958269, 0.834218, 0.782016, 0.725853,
                  0.660369, 0.623775, 0.587952, 0.634187, 0.525539,
                  0.503068, 0.516293, 0.485166, 0.499607, 0.514354,
                  0.516399, 0.491443, 0.452487, 0.433715, 0.431894,
                  0.433852, 0.426308, 0.426420, 0.435224, 0.420108,
                  0.414563, 0.408872, 0.418126, 0.415748, 0.416119,
                  0.408963, 0.416059, 0.415985, 0.415391, 0.415321]

val_accuracies_p4b = [0.221675, 0.379310, 0.566502, 0.640394, 0.640394,
                      0.699507, 0.773399, 0.827586, 0.822660, 0.822660,
                      0.847291, 0.852217, 0.857143, 0.822660, 0.871921,
                      0.876847, 0.871921, 0.881773, 0.871921, 0.862069,
                      0.866995, 0.862069, 0.876847, 0.901478, 0.891626,
                      0.896552, 0.891626, 0.896552, 0.896552, 0.896552,
                      0.901478, 0.906404, 0.896552, 0.896552, 0.901478,
                      0.906404, 0.901478, 0.901478, 0.901478, 0.901478]

epochs_p4b = range(1, 41)
class_names = [f"Level {i}" for i in range(11)]

# Classification Report
print("\n=== PHASE 4B CLASSIFICATION REPORT ===")
predictions_p4b = trainer_v4b.predict(laser_test_aug)
y_pred_p4b = np.argmax(predictions_p4b.predictions, axis=1)
y_true_p4b = predictions_p4b.label_ids
print(classification_report(y_true_p4b, y_pred_p4b, target_names=class_names))

# Accuracy Graph
plt.figure(figsize=(10, 5))
plt.plot(epochs_p4b, val_accuracies_p4b, label='Validation Accuracy',
         marker='o', color='blue')
plt.axhline(y=0.98, color='r', linestyle='--', label='Target (98%)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Phase 4B — Validation Accuracy vs Target')
plt.ylim([0.1, 1.0])
plt.legend()
plt.grid(True)
plt.savefig('accuracy_graph_phase4b.png', dpi=150, bbox_inches='tight')
plt.show()

# Loss Curve
plt.figure(figsize=(10, 5))
plt.plot(epochs_p4b, train_losses_p4b, label='Training Loss',
         marker='o', color='blue')
plt.plot(epochs_p4b, val_losses_p4b, label='Validation Loss',
         marker='s', color='orange')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Phase 4B — Training and Validation Loss Curve')
plt.legend()
plt.grid(True)
plt.savefig('loss_curve_phase4b.png', dpi=150, bbox_inches='tight')
plt.show()

# Confusion Matrix
class_names_full = [f"Soil Moisture Level {i}" for i in range(11)]
plt.figure(figsize=(14, 12))
cm_p4b = confusion_matrix(y_true_p4b, y_pred_p4b)
disp = ConfusionMatrixDisplay(confusion_matrix=cm_p4b,
                               display_labels=class_names_full)
disp.plot(cmap='Blues', xticks_rotation=45)
plt.title('Phase 4B Confusion Matrix — Weighted Loss ViT Classifier')
plt.tight_layout()
plt.savefig('confusion_matrix_phase4b.png', dpi=150, bbox_inches='tight')
plt.show()
print("Phase 4B metrics saved!")

In [ ]:
import os
import yaml

# Check Master_Soil_Moisture structure
MASTER_DIR = '/kaggle/working/Master_Soil_Moisture'
SOURCE_DIR = '/kaggle/working/source_data'

print("Master dir exists:", os.path.exists(MASTER_DIR))
print("Source dir exists:", os.path.exists(SOURCE_DIR))

In [ ]:
# Step 21: Two-Stage Inference Pipeline + Annotated Images
import os
import random
import zipfile
import torch
import yaml
import numpy as np
from PIL import Image, ImageDraw

SOURCE_DIR = '/kaggle/working/source_data'
OUTPUT_DIR = '/kaggle/working/inference_phase4a'
ZIP_PATH   = '/kaggle/working/soil_moisture_phase4a_50.zip'
os.makedirs(OUTPUT_DIR, exist_ok=True)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_v4.to(device)
model_v4.eval()

mapping = {
    'soil-moisture-1.0': 'Level 1', 'soil-moisture-2.0': 'Level 2',
    'soil-moisture-3.0': 'Level 3', 'soil-moisture-5.0': 'Level 5',
    'soil-moisture-8.2': 'Level 8',
    '0': 'Level 0', '1': 'Level 1', '2': 'Level 2', '3': 'Level 3',
    '4': 'Level 4', '5': 'Level 5', '6': 'Level 6', '7': 'Level 7',
    '8': 'Level 8', '9': 'Level 9', '10': 'Level 10',
}

samples_per_dataset = {
    'soil-moisture-v4':             8,
    'soil-moisture-v4-ir':          7,
    'soil-moisture-v4-uv':          7,
    'soil-moisture-ir':             7,
    'soil-moisture-5sagf':          7,
    'soil_moisture_september':      7,
    'soil_moisture_stir_september': 7,
}

def draw_bbox(img, x_center, y_center, width, height, color=(0, 255, 0)):
    """Draw bounding box on image."""
    W, H   = img.size
    x1     = int((x_center - width/2)  * W)
    y1     = int((y_center - height/2) * H)
    x2     = int((x_center + width/2)  * W)
    y2     = int((y_center + height/2) * H)
    draw   = ImageDraw.Draw(img)
    draw.rectangle([x1, y1, x2, y2], outline=color, width=3)
    return img

def annotate_image(img, dataset_name, img_id, pred_label, 
                   true_label, x_center, y_center, width, height):
    # Draw bounding box on original image
    img  = img.convert("RGB")
    img  = draw_bbox(img, x_center, y_center, width, height)
    img  = img.resize((640, 580))
    draw = ImageDraw.Draw(img)

    # Info panel
    panel = Image.new("RGB", (640, 780), (0, 0, 0))
    panel_draw = ImageDraw.Draw(panel)
    panel.paste(img, (0, 160))
    draw = ImageDraw.Draw(panel)

    draw.text((10, 8),  f"Dataset: {dataset_name}",   fill=(255, 255, 255))
    img_id_display = img_id[:50] + '...' if len(img_id) > 50 else img_id
    draw.text((10, 35), f"Image ID: {img_id_display}", fill=(255, 255, 255))
    pred_color = (0, 255, 0) if pred_label == true_label else (255, 0, 0)
    draw.text((10, 62), f"Predicted:    {pred_label}",  fill=pred_color)
    draw.text((10, 89), f"Ground Truth: {true_label}",  fill=(255, 255, 0))
    draw.text((10, 116), "Stage 1: Laser Detected | Stage 2: Moisture Classified",
              fill=(0, 200, 255))

    result_text  = "CORRECT" if pred_label == true_label else "INCORRECT"
    result_color = (0, 255, 0) if pred_label == true_label else (255, 0, 0)
    draw.rectangle([0, 740, 640, 780], fill=(0, 0, 0))
    draw.text((10, 748), result_text, fill=result_color)

    return panel

# Run two-stage inference
all_saved      = []
sample_counter = 1

for dataset_name, count in samples_per_dataset.items():
    dataset_path = os.path.join(SOURCE_DIR, dataset_name)
    if not os.path.exists(dataset_path):
        print(f"Skipping {dataset_name} — folder not found")
        continue

    img_dir = os.path.join(dataset_path, 'test', 'images')
    lbl_dir = os.path.join(dataset_path, 'test', 'labels')
    if not os.path.exists(img_dir):
        img_dir = os.path.join(dataset_path, 'valid', 'images')
        lbl_dir = os.path.join(dataset_path, 'valid', 'labels')
    if not os.path.exists(img_dir):
        print(f"No images found for {dataset_name}")
        continue

    yaml_path = os.path.join(dataset_path, 'data.yaml')
    with open(yaml_path, 'r') as f:
        class_names = yaml.safe_load(f)['names']

    all_imgs = [f for f in os.listdir(img_dir)
                if f.endswith(('.jpg', '.jpeg', '.png'))]
    selected = random.sample(all_imgs, min(count, len(all_imgs)))

    for img_file in selected:
        img_path = os.path.join(img_dir, img_file)
        lbl_path = os.path.join(lbl_dir, img_file.rsplit('.', 1)[0] + '.txt')

        # Get ground truth and bounding box
        true_label = 'Unknown'
        x_center = y_center = 0.5
        width = height = 1.0

        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as f:
                lines = f.readlines()
            if lines:
                parts      = lines[0].strip().split()
                class_id   = int(parts[0])
                x_center   = float(parts[1])
                y_center   = float(parts[2])
                width      = float(parts[3])
                height     = float(parts[4])
                raw_name   = str(class_names[class_id])
                true_label = mapping.get(raw_name, raw_name)

        # Stage 1: Crop laser region
        img     = Image.open(img_path).convert("RGB")
        cropped = crop_laser(img, x_center, y_center, width, height)

        # Stage 2: Classify cropped region
        inputs  = processor(images=cropped, return_tensors="pt")
        inputs  = {k: v.to(device) for k, v in inputs.items()}

        with torch.no_grad():
            outputs = model_v3(**inputs)

        pred_id    = outputs.logits.argmax(-1).item()
        pred_label = f"Level {pred_id}"

        # Annotate original image with bbox and prediction
        img_id    = img_file.rsplit('.', 1)[0]
        annotated = annotate_image(img, dataset_name, img_id, pred_label,
                                   true_label, x_center, y_center, 
                                   width, height)
        save_name = f"sample_{sample_counter:02d}_{dataset_name}.jpg"
        save_path = os.path.join(OUTPUT_DIR, save_name)
        annotated.save(save_path)
        all_saved.append(save_path)
        print(f"Sample {sample_counter:02d} | {dataset_name} | {img_id} | "
              f"Pred: {pred_label} | Truth: {true_label}")
        sample_counter += 1

# Zip results
with zipfile.ZipFile(ZIP_PATH, 'w') as zipf:
    for file_path in all_saved:
        zipf.write(file_path, os.path.basename(file_path))

print(f"\n{len(all_saved)} images saved and zipped!")
print(f"ZIP: {ZIP_PATH}")

from IPython.display import FileLink
display(FileLink(ZIP_PATH))

In [ ]:
# Step 22: Install YOLOv8
!pip install ultralytics -q

import ultralytics
ultralytics.checks()
print("YOLOv8 ready!")

In [ ]:
# Step 23: Prepare YOLOv8 Dataset
import os
import shutil
import yaml

SOURCE_DIR = '/kaggle/working/source_data'
YOLO_DIR   = '/kaggle/working/Master_YOLO'

# Class mapping — numeric only
mapping = {
    'soil-moisture-1.0': '1', 'soil-moisture-2.0': '2',
    'soil-moisture-3.0': '3', 'soil-moisture-5.0': '5',
    'soil-moisture-8.2': '8',
    '0': '0', '1': '1', '2': '2', '3': '3', '4': '4',
    '5': '5', '6': '6', '7': '7', '8': '8', '9': '9', '10': '10',
    # Level_X format from september datasets
    'Level_0': '0', 'Level_1': '1', 'Level_2': '2', 'Level_3': '3',
    'Level_4': '4', 'Level_5': '5', 'Level_6': '6', 'Level_7': '7',
    'Level_8': '8', 'Level_9': '9', 'Level_10': '10',
}

if os.path.exists(YOLO_DIR):
    shutil.rmtree(YOLO_DIR)

for split in ['train', 'valid', 'test']:
    os.makedirs(os.path.join(YOLO_DIR, split, 'images'), exist_ok=True)
    os.makedirs(os.path.join(YOLO_DIR, split, 'labels'), exist_ok=True)

skipped = 0
copied  = 0

for proj_folder in os.listdir(SOURCE_DIR):
    proj_path = os.path.join(SOURCE_DIR, proj_folder)
    yaml_path = os.path.join(proj_path, 'data.yaml')
    if not os.path.exists(yaml_path):
        continue

    with open(yaml_path, 'r') as f:
        class_names = yaml.safe_load(f)['names']

    for split in ['train', 'valid', 'test']:
        img_dir = os.path.join(proj_path, split, 'images')
        lbl_dir = os.path.join(proj_path, split, 'labels')
        target_split = 'valid' if split == 'valid' else split

        if not os.path.exists(img_dir):
            continue

        for img_file in os.listdir(img_dir):
            if not img_file.endswith(('.jpg', '.jpeg', '.png')):
                continue

            lbl_file = img_file.rsplit('.', 1)[0] + '.txt'
            lbl_path = os.path.join(lbl_dir, lbl_file)

            if not os.path.exists(lbl_path):
                skipped += 1
                continue

            with open(lbl_path, 'r') as f:
                lines = f.readlines()
            if not lines:
                skipped += 1
                continue

            # Remap class ID
            new_lines = []
            valid = True
            for line in lines:
                parts    = line.strip().split()
                class_id = int(parts[0])
                raw_name = str(class_names[class_id])
                clean    = mapping.get(raw_name, None)
                if clean is None:
                    valid = False
                    break
                new_line = f"{clean} {' '.join(parts[1:])}\n"
                new_lines.append(new_line)

            if not valid:
                skipped += 1
                continue

            # Copy image
            unique_name = f"{proj_folder}_{img_file}"
            shutil.copy(
                os.path.join(img_dir, img_file),
                os.path.join(YOLO_DIR, target_split, 'images', unique_name)
            )

            # Write remapped label
            lbl_unique = unique_name.rsplit('.', 1)[0] + '.txt'
            with open(os.path.join(YOLO_DIR, target_split, 'labels', lbl_unique), 'w') as f:
                f.writelines(new_lines)

            copied += 1

print(f"Dataset prepared! {copied} images copied, {skipped} skipped")

# Count per split
for split in ['train', 'valid', 'test']:
    img_path = os.path.join(YOLO_DIR, split, 'images')
    if os.path.exists(img_path):
        print(f"{split}: {len(os.listdir(img_path))} images")

In [ ]:
# Step 24: Create YOLOv8 data.yaml
import yaml

data_yaml = {
    'path': YOLO_DIR,
    'train': 'train/images',
    'val':   'valid/images',
    'test':  'test/images',
    'nc':    11,
    'names': {i: f"Level_{i}" for i in range(11)}
}

yaml_path = os.path.join(YOLO_DIR, 'data.yaml')
with open(yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print("data.yaml created!")
print(f"Location: {yaml_path}")

# Verify
with open(yaml_path, 'r') as f:
    print(yaml.safe_load(f))

In [ ]:
# Step 25: Train YOLOv8
from ultralytics import YOLO

# Load pretrained YOLOv8 small model
model_yolo = YOLO('yolov8s.pt')

# Train
results = model_yolo.train(
    data=os.path.join(YOLO_DIR, 'data.yaml'),
    epochs=50,
    imgsz=640,
    batch=16,
    name='soil_moisture_yolo',
    project='/kaggle/working/yolo_results',
    exist_ok=True,
    patience=10,         # Phase 6 — no augmentation
    save=True,
    plots=True,
    device=0,
    workers=2,
    lr0=0.001,
    weight_decay=0.0005,
    label_smoothing=0.1,
    val=True,
)

print("YOLOv8 training complete!")
print(f"Best model saved at: {results.save_dir}")

In [ ]:
# Step 26: Phase 1 — Regenerate Accuracy and Loss Curve plots
import matplotlib.pyplot as plt

train_losses_p1 = [2.344809, 2.118603, 1.936597, 1.603722, 1.359488,
                   1.207903, 1.005909, 0.903221, 0.784347, 0.738066,
                   0.660259, 0.600291, 0.564556, 0.526923, 0.508441,
                   0.523409, 0.499586]

val_losses_p1 = [2.318297, 2.081824, 1.797832, 1.574416, 1.370550,
                 1.195082, 1.046510, 0.922475, 0.825795, 0.748198,
                 0.666923, 0.640123, 0.605296, 0.580547, 0.567832,
                 0.554361, 0.547682]

val_accuracies_p1 = [0.270936, 0.448276, 0.674877, 0.862069, 0.871921,
                     0.886700, 0.940887, 0.950739, 0.950739, 0.955665,
                     0.960591, 0.960591, 0.960591, 0.965517, 0.965517,
                     0.965517, 0.965517]

epochs_p1 = range(1, 18)

# Accuracy Graph
plt.figure(figsize=(10, 5))
plt.plot(epochs_p1, val_accuracies_p1, label='Validation Accuracy',
         marker='o', color='blue')
plt.axhline(y=0.98, color='r', linestyle='--', label='Target (98%)')
plt.xlabel('Epoch')
plt.ylabel('Accuracy')
plt.title('Phase 1 — Validation Accuracy vs Target')
plt.ylim([0.2, 1.0])
plt.legend()
plt.grid(True)
plt.savefig('/kaggle/working/accuracy_graph_phase1.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Phase 1 accuracy graph saved!")

# Loss Curve
plt.figure(figsize=(10, 5))
plt.plot(epochs_p1, train_losses_p1, label='Training Loss',
         marker='o', color='blue')
plt.plot(epochs_p1, val_losses_p1, label='Validation Loss',
         marker='s', color='orange')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Phase 1 — Training and Validation Loss Curve')
plt.legend()
plt.grid(True)
plt.savefig('/kaggle/working/loss_curve_phase1.png',
            dpi=150, bbox_inches='tight')
plt.show()
print("Phase 1 loss curve saved!")

# Classification Report — Phase 1 hardcoded values
import numpy as np
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay

# Phase 1 test predictions — hardcoded from saved output
y_true_p1 = [0,0,0,0,0,0,0,0,
             1,1,1,1,1,1,1,1,1,1,1,1,
             2,2,2,2,2,2,2,2,2,
             3,3,3,3,3,3,3,3,3,3,3,3,3,3,3,
             4,4,4,4,4,4,4,4,4,
             5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,5,
             6,6,6,6,6,6,6,6,
             7,7,7,7,7,7,7,7,7,7,
             8,8,8,8,8,8,
             9,9,9,9,9,9,9,
             10,10,10,10,10,10]

# Phase 1 actual results from classification report
# precision recall f1 per class
class_names = [f"Level {i}" for i in range(11)]
print("\n=== PHASE 1 CLASSIFICATION REPORT ===")
print(f"{'':15} {'precision':>10} {'recall':>10} {'f1-score':>10} {'support':>10}")
print(f"{'Level 0':15} {'1.00':>10} {'1.00':>10} {'1.00':>10} {'8':>10}")
print(f"{'Level 1':15} {'1.00':>10} {'0.92':>10} {'0.96':>10} {'12':>10}")
print(f"{'Level 2':15} {'0.53':>10} {'1.00':>10} {'0.69':>10} {'9':>10}")
print(f"{'Level 3':15} {'1.00':>10} {'1.00':>10} {'1.00':>10} {'15':>10}")
print(f"{'Level 4':15} {'1.00':>10} {'0.89':>10} {'0.94':>10} {'9':>10}")
print(f"{'Level 5':15} {'1.00':>10} {'0.88':>10} {'0.93':>10} {'16':>10}")
print(f"{'Level 6':15} {'1.00':>10} {'1.00':>10} {'1.00':>10} {'8':>10}")
print(f"{'Level 7':15} {'1.00':>10} {'1.00':>10} {'1.00':>10} {'10':>10}")
print(f"{'Level 8':15} {'1.00':>10} {'0.83':>10} {'0.91':>10} {'6':>10}")
print(f"{'Level 9':15} {'1.00':>10} {'0.86':>10} {'0.92':>10} {'7':>10}")
print(f"{'Level 10':15} {'1.00':>10} {'0.67':>10} {'0.80':>10} {'6':>10}")
print(f"\n{'accuracy':15} {'':>10} {'':>10} {'0.92':>10} {'106':>10}")
print(f"{'macro avg':15} {'0.96':>10} {'0.91':>10} {'0.92':>10} {'106':>10}")
print(f"{'weighted avg':15} {'0.96':>10} {'0.92':>10} {'0.93':>10} {'106':>10}")

print("\nPhase 1 plots saved to /kaggle/working/")
print("Download: accuracy_graph_phase1.png and loss_curve_phase1.png")

In [ ]:
# Step 26: Phase 7 - Collect YOLO auto-generated metrics
import os
import shutil
import zipfile
from IPython.display import FileLink

yolo_results = '/kaggle/working/yolo_results/soil_moisture_yolo'
output_dir = '/kaggle/working/phase7_metrics'
zip_path = '/kaggle/working/phase7_metrics.zip'

# Create output directory FIRST before any file operations
os.makedirs(output_dir, exist_ok=True)

# List all auto-generated files
print('Available Phase 7 metric files:')
for root, dirs, files in os.walk(yolo_results):
    for file in files:
        print(os.path.join(root, file))

# Copy key metric files
key_files = [
    'confusion_matrix.png',
    'confusion_matrix_normalized.png',
    'results.png',
    'PR_curve.png',
    'F1_curve.png',
    'val_batch0_pred.jpg',
    'val_batch1_pred.jpg',
    'val_batch2_pred.jpg',
    'results.csv',
]

print(f"\nCopying key files to {output_dir}/")
for file in key_files:
    src = os.path.join(yolo_results, file)
    dst = os.path.join(output_dir, f'phase7_{file}')
    if os.path.exists(src):
        shutil.copy(src, dst)
        print(f'Copied: {file} -> phase7_{file}')
    else:
        print(f'Not found: {file}')

# Zip — directory now guaranteed to exist
with zipfile.ZipFile(zip_path, 'w') as zipf:
    for file in os.listdir(output_dir):
        zipf.write(os.path.join(output_dir, file), arcname=file)

print(f'\nAll Phase 7 metrics zipped at: {zip_path}')
display(FileLink(zip_path))

In [ ]:

# Step 27: Phase 6 — Two-Stage Inference Pipeline + Annotated Images
import os
import random
import zipfile
import yaml
from PIL import Image, ImageDraw, ImageFont
from ultralytics import YOLO

SOURCE_DIR = '/kaggle/working/source_data'
OUTPUT_DIR = '/kaggle/working/inference_phase6B'
ZIP_PATH   = '/kaggle/working/soil_moisture_phase6B_50.zip'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load best Phase 6 trained model
best_model_path = '/kaggle/working/yolo_results/soil_moisture_yolo/weights/best.pt'
model_yolo_inf  = YOLO(best_model_path)
model_class_names = model_yolo_inf.names

SEPTEMBER_DATASETS = {'soil_moisture_september', 'soil_moisture_stir_september'}

SOIL_MOISTURE_MAP = {
    'soil-moisture-1.0': '1',
    'soil-moisture-2.0': '2',
    'soil-moisture-3.0': '3',
    'soil-moisture-5.0': '5',
    'soil-moisture-8.2': '8',
}

samples_per_dataset = {
    'soil-moisture-v4':             8,
    'soil-moisture-v4-ir':          7,
    'soil-moisture-v4-uv':          7,
    'soil-moisture-ir':             7,
    'soil-moisture-5sagf':          7,
    'soil_moisture_september':      7,
    'soil_moisture_stir_september': 7,
}

# Load fonts
try:
    font_large  = ImageFont.truetype(
        "/usr/share/fonts/truetype/liberation/LiberationSans-Bold.ttf", 36)
    font_normal = ImageFont.truetype(
        "/usr/share/fonts/truetype/liberation/LiberationSans-Regular.ttf", 28)
except:
    font_large  = ImageFont.load_default()
    font_normal = ImageFont.load_default()

def annotate_yolo(img, dataset_name, img_id, pred_label,
                  true_label, bbox, conf):
    img  = img.convert("RGB")
    W, H = img.size

    draw = ImageDraw.Draw(img)
    if bbox is not None:
        x1, y1, x2, y2 = bbox
        pred_color = (0, 255, 0) if pred_label == true_label else (255, 0, 0)
        draw.rectangle([x1, y1, x2, y2], outline=pred_color, width=3)
        draw.text((x1, y2 + 5),
                  f"{pred_label} ({conf:.2f})",
                  fill=pred_color, font=font_large)

    img   = img.resize((640, 480))
    panel = Image.new("RGB", (640, 750), (0, 0, 0))
    panel.paste(img, (0, 200))
    draw  = ImageDraw.Draw(panel)

    line_height = font_large.size + 8
    y = 25
    draw.text((10, y), f"Dataset: {dataset_name}",
              fill=(255, 255, 255), font=font_large)
    y += line_height
    draw.text((10, y),
              f"Image ID: {img_id[:20]}{'...' if len(img_id) > 20 else ''}",
              fill=(255, 255, 255), font=font_large)
    y += line_height
    pred_color = (0, 255, 0) if pred_label == true_label else (255, 0, 0)
    draw.text((10, y), f"Predicted: {pred_label}",
              fill=pred_color, font=font_large)
    y += line_height
    draw.text((10, y), f"Ground Truth: {true_label}",
              fill=(255, 255, 0), font=font_large)
    y += line_height + 30
    conf_text = f"Confidence: {conf:.2f}" if conf > 0 else "No detection"
    draw.text((10, y), conf_text,
              fill=(0, 200, 255), font=font_large)

    result_text  = "CORRECT" if pred_label == true_label else "INCORRECT"
    result_color = (0, 255, 0) if pred_label == true_label else (255, 0, 0)
    draw.rectangle([0, 690, 640, 750], fill=(0, 0, 0))
    draw.text((10, 698), result_text, fill=result_color, font=font_large)

    return panel

# Run YOLO inference
all_saved      = []
sample_counter = 1

for dataset_name, count in samples_per_dataset.items():
    dataset_path = os.path.join(SOURCE_DIR, dataset_name)
    if not os.path.exists(dataset_path):
        print(f"Skipping {dataset_name} — not found")
        continue

    img_dir = os.path.join(dataset_path, 'test', 'images')
    lbl_dir = os.path.join(dataset_path, 'test', 'labels')
    if not os.path.exists(img_dir):
        img_dir = os.path.join(dataset_path, 'valid', 'images')
        lbl_dir = os.path.join(dataset_path, 'valid', 'labels')
    if not os.path.exists(img_dir):
        continue

    yaml_path = os.path.join(dataset_path, 'data.yaml')
    with open(yaml_path, 'r') as f:
        class_names = yaml.safe_load(f)['names']

    all_imgs = [f for f in os.listdir(img_dir)
                if f.endswith(('.jpg', '.jpeg', '.png'))]
    selected = random.sample(all_imgs, min(count, len(all_imgs)))

    for img_file in selected:
        img_path = os.path.join(img_dir, img_file)
        lbl_path = os.path.join(lbl_dir, img_file.rsplit('.', 1)[0] + '.txt')

        # Get ground truth
        true_label = 'Unknown'
        if os.path.exists(lbl_path):
            with open(lbl_path, 'r') as f:
                lines = f.readlines()
            if lines:
                parts    = lines[0].strip().split()
                class_id = int(parts[0])
                raw_name = str(class_names[class_id]).replace('_', ' ')

                if raw_name.isdigit():
                    true_label = f"Level {raw_name}"
                elif raw_name.startswith('soil-moisture-'):
                    mapped = SOIL_MOISTURE_MAP.get(
                                raw_name.replace(' ', '-'), None)
                    if mapped:
                        true_label = f"Level {mapped}"
                    else:
                        num = raw_name.replace('soil-moisture-', '')\
                                      .replace('.0', '')
                        true_label = f"Level {num}"
                else:
                    true_label = raw_name

        # Run YOLO inference
        img     = Image.open(img_path).convert("RGB")
        results = model_yolo_inf(img_path, verbose=False)

        pred_label = 'No Detection'
        bbox       = None
        conf       = 0.0

        if len(results[0].boxes) > 0:
            boxes      = results[0].boxes
            best_idx   = boxes.conf.argmax().item()
            pred_id    = int(boxes.cls[best_idx].item())
            conf       = float(boxes.conf[best_idx].item())
            raw_pred   = model_class_names.get(pred_id, str(pred_id))
            pred_label = raw_pred.replace('_', ' ')
            xyxy       = boxes.xyxy[best_idx].cpu().numpy()
            bbox       = (int(xyxy[0]), int(xyxy[1]),
                         int(xyxy[2]), int(xyxy[3]))

        # Annotate and save
        img_id    = img_file.rsplit('.', 1)[0]
        annotated = annotate_yolo(img, dataset_name, img_id,
                                  pred_label, true_label, bbox, conf)
        save_name = f"sample_{sample_counter:02d}_{dataset_name}.jpg"
        save_path = os.path.join(OUTPUT_DIR, save_name)
        annotated.save(save_path)
        all_saved.append(save_path)
        print(f"Sample {sample_counter:02d} | {dataset_name} | "
              f"Pred: {pred_label} ({conf:.2f}) | Truth: {true_label}")
        sample_counter += 1

# Zip results
with zipfile.ZipFile(ZIP_PATH, 'w') as zipf:
    for file_path in all_saved:
        zipf.write(file_path, os.path.basename(file_path))

print(f"\n{len(all_saved)} images saved and zipped!")
print(f"ZIP: {ZIP_PATH}")

from IPython.display import FileLink
display(FileLink(ZIP_PATH))

In [ ]:
import os

yolo_results = '/kaggle/working/yolo_results/soil_moisture_yolo'

for root, dirs, files in os.walk(yolo_results):
    for file in files:
        print(os.path.join(root, file))

In [ ]:
import os

SOURCE_DIR = '/kaggle/working/source_data'

print("Images per dataset:")
for proj_folder in sorted(os.listdir(SOURCE_DIR)):
    proj_path = os.path.join(SOURCE_DIR, proj_folder)
    total = 0
    for split in ['train', 'valid', 'test']:
        split_path = os.path.join(proj_path, split, 'images')
        if os.path.exists(split_path):
            count = len(os.listdir(split_path))
            total += count
    print(f"  {proj_folder}: {total} images")

In [ ]:
import os

MASTER_DIR = '/kaggle/working/Master_Soil_Moisture'

print("=== All files in Master_Soil_Moisture ===")
for split in ['train', 'validation', 'test']:
    split_path = os.path.join(MASTER_DIR, split)
    if not os.path.exists(split_path):
        print(f"{split}: NOT FOUND")
        continue
    print(f"\n{split}:")
    for class_folder in sorted(os.listdir(split_path)):
        folder_path = os.path.join(split_path, class_folder)
        files = os.listdir(folder_path)
        sep_files = [f for f in files if 'september' in f]
        print(f"  {class_folder}: {len(files)} total, {len(sep_files)} from september")

In [ ]:
import os
import yaml

SOURCE_DIR = '/kaggle/working/source_data'

# Check soil-moisture-ir label files and class names
dataset_path = os.path.join(SOURCE_DIR, 'soil-moisture-ir')
yaml_path = os.path.join(dataset_path, 'data.yaml')

with open(yaml_path, 'r') as f:
    class_names = yaml.safe_load(f)['names']

print("soil-moisture-ir class names:")
for i, name in enumerate(class_names):
    print(f"  index {i} -> '{name}'")

# Check test label files
img_dir = os.path.join(dataset_path, 'test', 'images')
lbl_dir = os.path.join(dataset_path, 'test', 'labels')
if not os.path.exists(img_dir):
    img_dir = os.path.join(dataset_path, 'valid', 'images')
    lbl_dir = os.path.join(dataset_path, 'valid', 'labels')

print("\nSample label files:")
for f in os.listdir(lbl_dir)[:10]:
    with open(os.path.join(lbl_dir, f), 'r') as lf:
        first_line = lf.readline().strip()
    class_id = int(first_line.split()[0])
    print(f"  {f}: class_id={class_id} -> '{class_names[class_id]}'")

In [ ]:
import os

MASTER_DIR = '/kaggle/working/Master_Soil_Moisture'

# Show exact folder order as HuggingFace sees it
folders = sorted(os.listdir(os.path.join(MASTER_DIR, 'train')))
print("HuggingFace alphabetical index assignment:")
for idx, folder in enumerate(folders):
    print(f"  idx {idx} -> folder '{folder}' (should be class {folder})")

In [ ]:
import os

YOLO_DIR = '/kaggle/working/Master_YOLO'

# Check how many september images are in each class folder
print("=== September images in YOLO training data ===")
for split in ['train', 'valid', 'test']:
    split_path = os.path.join(YOLO_DIR, split, 'images')
    if not os.path.exists(split_path):
        continue
    sep_files = [f for f in os.listdir(split_path) if 'september' in f]
    print(f"\n{split}: {len(sep_files)} september images")
    for f in sep_files[:5]:
        print(f"  {f}")

# Also check corresponding label files
print("\n=== Sample label content for september images ===")
for split in ['train', 'valid', 'test']:
    lbl_path = os.path.join(YOLO_DIR, split, 'labels')
    if not os.path.exists(lbl_path):
        continue
    sep_lbls = [f for f in os.listdir(lbl_path) if 'september' in f]
    for f in sep_lbls[:3]:
        with open(os.path.join(lbl_path, f), 'r') as lf:
            content = lf.readline().strip()
        print(f"  {split}/{f}: {content}")

In [ ]:
import os
SOURCE_DIR = '/kaggle/working/source_data'
img_dir = os.path.join(SOURCE_DIR, 'soil_moisture_stir_september', 'test', 'images')
if not os.path.exists(img_dir):
    img_dir = os.path.join(SOURCE_DIR, 'soil_moisture_stir_september', 'valid', 'images')
print(f"Available test images: {len(os.listdir(img_dir))}")
for f in os.listdir(img_dir):
    print(f)

In [ ]:
#Step 27 Extension: Investigation of the September Datasets
import os
import random
import matplotlib.pyplot as plt
from PIL import Image

SOURCE_DIR = '/kaggle/working/source_data'

datasets_to_compare = [
    'soil_moisture_stir_september',
    'soil_moisture_september',
    'soil-moisture-v4-uv',
    'soil-moisture-ir',
]

fig, axes = plt.subplots(len(datasets_to_compare), 3, figsize=(15, 20))

for row, dataset in enumerate(datasets_to_compare):
    for split in ['test', 'valid', 'train']:
        img_dir = os.path.join(SOURCE_DIR, dataset, split, 'images')
        if os.path.exists(img_dir):
            break

    imgs = [f for f in os.listdir(img_dir)
            if f.endswith(('.jpg', '.jpeg', '.png'))]
    selected = random.sample(imgs, min(3, len(imgs)))

    for col, img_file in enumerate(selected):
        img = Image.open(os.path.join(img_dir, img_file)).convert("RGB")
        axes[row, col].imshow(img)
        axes[row, col].set_title(f"{dataset}\n{img_file[:30]}", fontsize=8)
        axes[row, col].axis('off')

plt.tight_layout()
plt.savefig('/kaggle/working/laser_pattern_comparison.png', dpi=150)
plt.show()
print("Saved: laser_pattern_comparison.png")

In [ ]:
# Figure 2 — Phase 6 Per-Class mAP50 Bar Chart
import os
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

OUTPUT_DIR = '/kaggle/working/paper_figures'
os.makedirs(OUTPUT_DIR, exist_ok=True)

classes = ['Level_0','Level_1','Level_2','Level_3','Level_4','Level_5',
           'Level_6','Level_7','Level_8','Level_9','Level_10']
map50   = [92.7, 89.1, 86.5, 89.4, 97.2, 95.4, 94.6, 93.6, 91.1, 70.2, 75.9]
overall_map50 = 95.3

# Color assignment
colors = []
for i, v in enumerate(map50):
    if v == max(map50):
        colors.append('#BA7517')  # gold for highest
    elif v == min(map50):
        colors.append('#E24B4A')  # red for lowest
    else:
        colors.append('#1D9E75')  # teal for all others

fig, ax = plt.subplots(figsize=(14, 7))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

bars = ax.bar(classes, map50, color=colors, width=0.6, zorder=3)

# Bar value labels
for bar, val in zip(bars, map50):
    ax.text(bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.3,
            f'{val}%',
            ha='center', va='bottom',
            fontsize=9, fontname='DejaVu Sans', fontweight='bold')

# Overall reference line
ax.axhline(overall_map50, color='#185FA5', linestyle='--',
           linewidth=1.5, zorder=4)
ax.text(len(classes) - 0.4, overall_map50 + 0.3,
        f'Overall\n{overall_map50}%',
        color='#185FA5', fontsize=8.5,
        fontname='DejaVu Sans', ha='right')

# Legend
legend_patches = [
    mpatches.Patch(color='#BA7517', label=f'Highest: Level_4 (97.2%)'),
    mpatches.Patch(color='#1D9E75', label='All other classes'),
    mpatches.Patch(color='#E24B4A', label=f'Lowest: Level_9 (70.2%)'),
]
ax.legend(handles=legend_patches, fontsize=9, loc='lower left',
          framealpha=0.9)

# Axes
ax.set_ylim(64, 101)
ax.set_xlabel('Moisture Level Class', fontsize=10, fontname='DejaVu Sans')
ax.set_ylabel('mAP50 (%)', fontsize=10, fontname='DejaVu Sans')
ax.set_title(
    'Phase 6 Per-Class mAP50 — YOLOv8 Corrected Annotations\n'
    '(Overall: 95.3% mAP50, 89.1% Inference Accuracy, 41/46 correct)',
    fontsize=11, fontname='DejaVu Sans', pad=12
)
ax.tick_params(axis='x', labelsize=9)
ax.tick_params(axis='y', labelsize=9)
ax.grid(axis='y', alpha=0.3, zorder=0)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig2_per_class_map50.png'),
            dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(os.path.join(OUTPUT_DIR, 'fig2_per_class_map50.pdf'),
            bbox_inches='tight', facecolor='white')
plt.close()
print("Figure 2 saved to /kaggle/output/paper_figures/")

import shutil
from IPython.display import FileLink, display

# Copy to working directory for download
shutil.copy('/kaggle/output/paper_figures/fig2_per_class_map50.png',
            '/kaggle/working/fig2_per_class_map50.png')
shutil.copy('/kaggle/output/paper_figures/fig2_per_class_map50.pdf',
            '/kaggle/working/fig2_per_class_map50.pdf')

display(FileLink('/kaggle/working/fig2_per_class_map50.png'))
display(FileLink('/kaggle/working/fig2_per_class_map50.pdf'))

In [ ]:
# Figure 3 — Cross-Dataset Inference Accuracy: Phase 5 vs Phase 6 vs Phase 7
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import shutil
from IPython.display import FileLink, display

OUTPUT_DIR = '/kaggle/output/paper_figures'
os.makedirs(OUTPUT_DIR, exist_ok=True)

datasets = ['soil-moisture-v4', 'soil-moisture-v4-ir', 'soil-moisture-v4-uv',
            'soil-moisture-ir', 'soil-moisture-5sagf',
            'soil_moisture_september', 'soil_moisture_stir_september']

short_labels = ['soil-moisture\n-v4', 'soil-moisture\n-v4-ir', 'soil-moisture\n-v4-uv',
                'soil-moisture\n-ir', 'soil-moisture\n-5sagf',
                'soil_moisture\n_september', 'soil_moisture\n_stir_sept']

phase5 = [100.0, 100.0, 100.0, 85.7, 100.0, 14.3, 71.4]
phase6 = [100.0, 100.0, 100.0, 100.0, 100.0, 57.1, 20.0]
phase7 = [100.0, 100.0, 100.0, 100.0, 100.0, 50.0, 20.0]

x     = np.arange(len(datasets))
width = 0.25

fig, ax = plt.subplots(figsize=(16, 8))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')

bars5 = ax.bar(x - width, phase5, width,
               label='Phase 5 (81.25% overall)', color='#185FA5', zorder=3)
bars6 = ax.bar(x,         phase6, width,
               label='Phase 6 — Production (89.1% overall)', color='#BA7517', zorder=3)
bars7 = ax.bar(x + width, phase7, width,
               label='Phase 7 (86.9% overall)', color='#1D9E75', zorder=3)

# Bar value labels
for bars in [bars5, bars6, bars7]:
    for bar in bars:
        height = bar.get_height()
        ax.text(bar.get_x() + bar.get_width() / 2,
                height + 0.8,
                f'{height:.0f}%',
                ha='center', va='bottom',
                fontsize=7.5, fontname='DejaVu Sans')

# Annotation — Phase 5 stir_september inflated by label error
# Points to the 71.4% blue bar (Phase 5, last dataset)
ax.annotate(
    'Phase 5 stir_september was\ninflated by label error',
    xy=(x[6] - width, 71.4),
    xytext=(x[6] - width - 1.8, 88),
    fontsize=8, fontname='DejaVu Sans', color='#E24B4A',
    arrowprops=dict(arrowstyle='->', color='#E24B4A', lw=1.5)
)

# Annotation — Annotation correction +42.8pp
# Points from Phase 5 september bar (14.3%) showing gain to Phase 6 (57.1%)
ax.annotate(
    'Annotation correction\n+42.8pp',
    xy=(x[5] - width, 14.3),
    xytext=(x[5] - width - 1.2, 40),
    fontsize=8, fontname='DejaVu Sans', color='#533AB7',
    arrowprops=dict(arrowstyle='->', color='#533AB7', lw=1.5)
)

# Axes
ax.set_ylim(0, 112)
ax.set_xticks(x)
ax.set_xticklabels(short_labels, fontsize=8.5, fontname='DejaVu Sans')
ax.set_xlabel('Source Dataset', fontsize=10, fontname='DejaVu Sans')
ax.set_ylabel('Inference Accuracy (%)', fontsize=10, fontname='DejaVu Sans')
ax.set_title(
    'Cross-Dataset Inference Accuracy: Phase 5 vs Phase 6 (Production) vs Phase 7',
    fontsize=11, fontname='DejaVu Sans', pad=12
)
ax.legend(fontsize=9, loc='lower right', framealpha=0.9)
ax.grid(axis='y', alpha=0.3, zorder=0)
ax.tick_params(axis='y', labelsize=9)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'fig3_cross_dataset_inference.png'),
            dpi=300, bbox_inches='tight', facecolor='white')
plt.savefig(os.path.join(OUTPUT_DIR, 'fig3_cross_dataset_inference.pdf'),
            bbox_inches='tight', facecolor='white')
plt.close()

# Copy to working for download
shutil.copy(os.path.join(OUTPUT_DIR, 'fig3_cross_dataset_inference.png'),
            '/kaggle/working/fig3_cross_dataset_inference.png')
shutil.copy(os.path.join(OUTPUT_DIR, 'fig3_cross_dataset_inference.pdf'),
            '/kaggle/working/fig3_cross_dataset_inference.pdf')

display(FileLink('/kaggle/working/fig3_cross_dataset_inference.png'))
display(FileLink('/kaggle/working/fig3_cross_dataset_inference.pdf'))
print("Figure 3 saved.")

In [ ]:
# Figure 5 — Phase 6 YOLOv8 Training Curves
import os
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import shutil
from IPython.display import FileLink, display

OUTPUT_DIR = '/kaggle/output/paper_figures'
os.makedirs(OUTPUT_DIR, exist_ok=True)

csv_path = '/kaggle/working/yolo_results/soil_moisture_yolo/results.csv'

if not os.path.exists(csv_path):
    print("ERROR: results.csv not found. Run Steps 4-25 first.")
else:
    df = pd.read_csv(csv_path)
    df.columns = df.columns.str.strip()
    print("Columns found:", list(df.columns))

    epochs     = df.index + 1
    best_epoch = 32

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))
    fig.patch.set_facecolor('white')

    # Panel 1 — Box Loss
    axes[0,0].plot(epochs, df['train/box_loss'],
                   label='Train', color='#185FA5', linewidth=1.5)
    axes[0,0].plot(epochs, df['val/box_loss'],
                   label='Val', color='#E24B4A', linewidth=1.5)
    axes[0,0].axvline(best_epoch, color='gray', linestyle='--',
                      alpha=0.7, label=f'Best epoch {best_epoch}')
    axes[0,0].set_title('Box Loss', fontsize=10, fontname='DejaVu Sans')
    axes[0,0].set_xlabel('Epoch', fontsize=9, fontname='DejaVu Sans')
    axes[0,0].set_ylabel('Loss', fontsize=9, fontname='DejaVu Sans')
    axes[0,0].legend(fontsize=8)
    axes[0,0].set_facecolor('white')
    axes[0,0].grid(alpha=0.3)

    # Panel 2 — Classification Loss
    axes[0,1].plot(epochs, df['train/cls_loss'],
                   label='Train', color='#185FA5', linewidth=1.5)
    axes[0,1].plot(epochs, df['val/cls_loss'],
                   label='Val', color='#E24B4A', linewidth=1.5)
    axes[0,1].axvline(best_epoch, color='gray', linestyle='--',
                      alpha=0.7, label=f'Best epoch {best_epoch}')
    axes[0,1].set_title('Classification Loss', fontsize=10,
                         fontname='DejaVu Sans')
    axes[0,1].set_xlabel('Epoch', fontsize=9, fontname='DejaVu Sans')
    axes[0,1].set_ylabel('Loss', fontsize=9, fontname='DejaVu Sans')
    axes[0,1].legend(fontsize=8)
    axes[0,1].set_facecolor('white')
    axes[0,1].grid(alpha=0.3)

    # Panel 3 — mAP50
    map50_col = 'metrics/mAP50(B)'
    axes[1,0].plot(epochs, df[map50_col],
                   color='#1D9E75', linewidth=2)
    axes[1,0].axvline(best_epoch, color='gray', linestyle='--', alpha=0.7)
    best_map = df[map50_col].max()
    best_map_epoch = df[map50_col].idxmax() + 1
    axes[1,0].annotate(
        f'Best: {best_map:.3f} (95.3%)',
        xy=(best_map_epoch, best_map),
        xytext=(best_map_epoch + 2, best_map - 0.06),
        fontsize=8, fontname='DejaVu Sans',
        arrowprops=dict(arrowstyle='->', color='black')
    )
    axes[1,0].set_title('mAP50', fontsize=10, fontname='DejaVu Sans')
    axes[1,0].set_xlabel('Epoch', fontsize=9, fontname='DejaVu Sans')
    axes[1,0].set_ylabel('mAP50', fontsize=9, fontname='DejaVu Sans')
    axes[1,0].set_facecolor('white')
    axes[1,0].grid(alpha=0.3)

    # Panel 4 — Precision and Recall
    axes[1,1].plot(epochs, df['metrics/precision(B)'],
                   label='Precision', color='#185FA5', linewidth=1.5)
    axes[1,1].plot(epochs, df['metrics/recall(B)'],
                   label='Recall', color='#E24B4A', linewidth=1.5)
    axes[1,1].axvline(best_epoch, color='gray', linestyle='--',
                      alpha=0.7, label=f'Best epoch {best_epoch}')
    axes[1,1].set_title('Precision and Recall', fontsize=10,
                         fontname='DejaVu Sans')
    axes[1,1].set_xlabel('Epoch', fontsize=9, fontname='DejaVu Sans')
    axes[1,1].set_ylabel('Value', fontsize=9, fontname='DejaVu Sans')
    axes[1,1].legend(fontsize=8)
    axes[1,1].set_facecolor('white')
    axes[1,1].grid(alpha=0.3)

    fig.suptitle(
        'Phase 6 YOLOv8 Training Curves — Best Epoch 32/42 '
        '(EarlyStopping patience=10)',
        fontsize=11, fontname='DejaVu Sans', y=1.02
    )

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'fig5_phase6_training_curves.png'),
                dpi=300, bbox_inches='tight', facecolor='white')
    plt.savefig(os.path.join(OUTPUT_DIR, 'fig5_phase6_training_curves.pdf'),
                bbox_inches='tight', facecolor='white')
    plt.close()

    # Copy to working for download
    shutil.copy(os.path.join(OUTPUT_DIR, 'fig5_phase6_training_curves.png'),
                '/kaggle/working/fig5_phase6_training_curves.png')
    shutil.copy(os.path.join(OUTPUT_DIR, 'fig5_phase6_training_curves.pdf'),
                '/kaggle/working/fig5_phase6_training_curves.pdf')

    display(FileLink('/kaggle/working/fig5_phase6_training_curves.png'))
    display(FileLink('/kaggle/working/fig5_phase6_training_curves.pdf'))
    print("Figure 5 saved.")

In [ ]:
# Figure 6 — Phase 6 Normalised Confusion Matrix
import os
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import shutil
from IPython.display import FileLink, display
from ultralytics import YOLO

OUTPUT_DIR = '/kaggle/output/paper_figures'
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load Phase 6 best model and run validation to get confusion matrix
model_path = '/kaggle/working/yolo_results/soil_moisture_yolo/weights/best.pt'
data_yaml  = '/kaggle/working/Master_YOLO/data.yaml'

if not os.path.exists(model_path):
    print("ERROR: best.pt not found. Run Steps 1-25 first.")
else:
    model   = YOLO(model_path)
    metrics = model.val(data=data_yaml, verbose=False)

    # Get confusion matrix from metrics
    cm       = metrics.confusion_matrix.matrix  # raw counts
    cm_float = cm.astype(float)

    # Normalise row-wise (true class totals)
    row_sums = cm_float.sum(axis=1, keepdims=True)
    row_sums[row_sums == 0] = 1  # avoid division by zero
    cm_norm  = cm_float / row_sums

    # Class labels — 11 moisture levels plus background
    n        = cm_norm.shape[0]
    if n == 12:
        labels = [f'Level_{i}' for i in range(11)] + ['background']
    else:
        labels = [f'Level_{i}' for i in range(n)]

    fig, ax = plt.subplots(figsize=(14, 12))
    fig.patch.set_facecolor('white')

    # Base colormap
    base_cmap = plt.cm.Blues
    im = ax.imshow(cm_norm, cmap=base_cmap, vmin=0, vmax=1)

    # Highlight diagonal with slightly different shade
    for i in range(n):
        ax.add_patch(plt.Rectangle(
            (i - 0.5, i - 0.5), 1, 1,
            fill=True,
            facecolor=plt.cm.YlOrBr(cm_norm[i, i] * 0.8 + 0.1),
            alpha=0.3,
            zorder=2
        ))

    # Annotate each cell
    for i in range(n):
        for j in range(n):
            val = cm_norm[i, j]
            if val > 0:
                text_color = 'white' if val > 0.6 else 'black'
                ax.text(j, i, f'{val:.2f}',
                        ha='center', va='center',
                        fontsize=7.5, color=text_color,
                        fontname='DejaVu Sans', zorder=3)

    # Colorbar
    plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

    # Labels
    ax.set_xticks(range(n))
    ax.set_yticks(range(n))
    ax.set_xticklabels(labels, rotation=45, ha='right',
                       fontsize=8, fontname='DejaVu Sans')
    ax.set_yticklabels(labels, fontsize=8, fontname='DejaVu Sans')
    ax.set_xlabel('Predicted Class', fontsize=10, fontname='DejaVu Sans')
    ax.set_ylabel('True Class', fontsize=10, fontname='DejaVu Sans')
    ax.set_title(
        'Phase 6 Normalised Confusion Matrix — YOLOv8 (95.3% mAP50)',
        fontsize=11, fontname='DejaVu Sans', pad=12
    )

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'fig6_confusion_matrix.png'),
                dpi=300, bbox_inches='tight', facecolor='white')
    plt.savefig(os.path.join(OUTPUT_DIR, 'fig6_confusion_matrix.pdf'),
                bbox_inches='tight', facecolor='white')
    plt.close()

    # Copy to working for download
    shutil.copy(os.path.join(OUTPUT_DIR, 'fig6_confusion_matrix.png'),
                '/kaggle/working/fig6_confusion_matrix.png')
    shutil.copy(os.path.join(OUTPUT_DIR, 'fig6_confusion_matrix.pdf'),
                '/kaggle/working/fig6_confusion_matrix.pdf')

    display(FileLink('/kaggle/working/fig6_confusion_matrix.png'))
    display(FileLink('/kaggle/working/fig6_confusion_matrix.pdf'))
    print("Figure 6 saved.")

In [ ]:
# Figure 4 (font fix) Visual Comparison
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import os
import glob
import shutil
import random
from PIL import Image
from IPython.display import FileLink, display

plt.rcParams.update({
    'font.size': 32,
    'axes.titlesize': 36,
    'axes.labelsize': 32,
    'xtick.labelsize': 26,
    'ytick.labelsize': 26,
    'legend.fontsize': 26,
    'lines.linewidth': 3.0,
    'axes.linewidth': 2.5,
    'font.family': 'Liberation sans',
})

OUT = '/kaggle/working/paper_figures_large'
os.makedirs(OUT, exist_ok=True)

def save(fig, name):
    for ext in ('pdf', 'png'):
        fig.savefig(f'{OUT}/{name}.{ext}', dpi=300,
                    bbox_inches='tight', facecolor='white')
    print(f'Saved {name}')
    plt.close(fig)
    
# ══════════════════════════════════════════════════════════════════════════════
# FIG 4 — Laser Pattern Visual Comparison
# ══════════════════════════════════════════════════════════════════════════════
print("Generating Figure 4...")

panel_info = [
    ('soil_moisture_stir_september', 'IR Laser\nUncontrolled Field',
     'Dim/invisible laser spot\nPhase 6 accuracy: 20%', 'red'),
    ('soil_moisture_september',      'UV Laser\nField',
     'Consistent blue UV glow\nPhase 6 accuracy: 57%', 'orange'),
    ('soil-moisture-v4-uv',          'UV Laser\nControlled Lab',
     'Strong, bright UV spot\nPhase 6 accuracy: 100%', '#2E7D5E'),
    ('soil-moisture-ir',             'IR Laser\nControlled Lab',
     'Clear white IR spot\nPhase 6 accuracy: 100%', '#2E7D5E'),
]

fig, axes = plt.subplots(1, 4, figsize=(40, 14))
fig.suptitle(
    'Laser-Pattern Visual Comparison:\nCapture Environment Governs Performance, Not Wavelength',
    fontsize=42, fontweight='bold', y=1.04
)

for ax, (ds, modality, caption, border_col) in zip(axes, panel_info):
    img_path = None
    for split in ['test', 'valid', 'train']:
        for ext in ['*.jpg', '*.png']:
            pattern = os.path.join(SOURCE_DIR, ds, split, 'images', ext)
            matches = glob.glob(pattern)
            if matches:
                img_path = matches[0]
                break
        if img_path:
            break

    if img_path:
        img = Image.open(img_path)
        ax.imshow(img)
    else:
        ax.set_facecolor('#222222')
        ax.text(0.5, 0.5, f'[{ds}]\nimage not found',
                ha='center', va='center', color='white',
                fontsize=20, transform=ax.transAxes)

    ax.set_title(f'{ds}\n{modality}', fontsize=32,
                 fontweight='bold', color=border_col, pad=16)
    ax.set_xlabel(caption, fontsize=30, color=border_col, labelpad=12)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_edgecolor(border_col)
        spine.set_linewidth(8)

fig.text(0.5, -0.05,
         'Left to right: IR uncontrolled field (20%), UV field (57%), '
         'UV controlled lab (100%), IR controlled lab (100%)',
         ha='center', fontsize=30, style='italic')
fig.tight_layout()
save(fig, 'fig4_laser_pattern_comparison')

In [ ]:
# Figure 7 — 7 separate files, one per dataset, 3 representative images each
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import numpy as np
import os
import glob
import shutil
from PIL import Image
from IPython.display import FileLink, display

plt.rcParams.update({
    'font.size': 14,
    'axes.titlesize': 14,
    'font.family': 'Liberation Sans',
})

OUT = '/kaggle/working/paper_figures_large'
os.makedirs(OUT, exist_ok=True)

def save(fig, name):
    for ext in ('pdf', 'png'):
        fig.savefig(f'{OUT}/{name}.{ext}', dpi=300,
                    bbox_inches='tight', facecolor='white')
    print(f'Saved {name}')
    plt.close(fig)

dataset_order = [
    'soil-moisture-v4',
    'soil-moisture-v4-ir',
    'soil-moisture-v4-uv',
    'soil-moisture-ir',
    'soil-moisture-5sagf',
    'soil_moisture_september',
    'soil_moisture_stir_september',
]

dataset_labels = {
    'soil-moisture-v4':
        'Dataset 1: soil-moisture-v4 (Standard RGB)',
    'soil-moisture-v4-ir':
        'Dataset 2: soil-moisture-v4-ir (Infrared)',
    'soil-moisture-v4-uv':
        'Dataset 3: soil-moisture-v4-uv (Ultraviolet)',
    'soil-moisture-ir':
        'Dataset 4: soil-moisture-ir (Infrared Controlled)',
    'soil-moisture-5sagf':
        'Dataset 5: soil-moisture-5sagf (General Field)',
    'soil_moisture_september':
        'Dataset 6: soil-moisture-september (Seasonal UV)',
    'soil_moisture_stir_september':
        'Dataset 7: soil-moisture-stir-september (IR Stirred)',
}

output_names = {
    'soil-moisture-v4':             'inference_soil_moisture_v4',
    'soil-moisture-v4-ir':          'inference_soil_moisture_v4_ir',
    'soil-moisture-v4-uv':          'inference_soil_moisture_v4_uv',
    'soil-moisture-ir':             'inference_soil_moisture_ir',
    'soil-moisture-5sagf':          'inference_soil_moisture_5sagf',
    'soil_moisture_september':      'inference_soil_moisture_september',
    'soil_moisture_stir_september': 'inference_soil_moisture_stir_september',
}

# Gather all inference images
all_files = sorted(
    glob.glob(f'{INFERENCE_DIR}/*.jpg') +
    glob.glob(f'{INFERENCE_DIR}/*.png')
)

# Group by dataset — match longest name first to avoid substring conflicts
grouped = {d: [] for d in dataset_order}
for f in all_files:
    fname = os.path.basename(f)
    for d in sorted(dataset_order, key=len, reverse=True):
        if d in fname:
            grouped[d].append(f)
            break

print('Generating Figure 7 — 7 separate dataset files...')

for d in dataset_order:
    files = grouped[d]
    if not files:
        print(f'No images found for {d} — skipping')
        continue

    # Detect correct/incorrect via green/red pixel counting on bottom strip
    correct   = []
    incorrect = []
    for f in files:
        img_arr      = np.array(Image.open(f))
        bottom_strip = img_arr[-60:, :, :]
        green_pixels = np.sum(
            (bottom_strip[:, :, 1] > 150) &
            (bottom_strip[:, :, 0] < 100) &
            (bottom_strip[:, :, 2] < 100)
        )
        red_pixels = np.sum(
            (bottom_strip[:, :, 0] > 150) &
            (bottom_strip[:, :, 1] < 100) &
            (bottom_strip[:, :, 2] < 100)
        )
        if green_pixels > red_pixels:
            correct.append(f)
        else:
            incorrect.append(f)

    print(f'{d}: {len(correct)} correct, {len(incorrect)} incorrect')

    # 2 correct + 1 incorrect, fill to 3 if needed
    picks = []
    if len(correct) >= 2:
        picks += correct[:2]
    else:
        picks += correct
    if len(incorrect) >= 1:
        picks += incorrect[:1]
    remaining = [f for f in files if f not in picks]
    while len(picks) < 3 and remaining:
        picks.append(remaining.pop(0))
    picks = picks[:3]

    ncols = len(picks)
    fig, axes = plt.subplots(1, ncols, figsize=(ncols * 7, 10))
    if ncols == 1:
        axes = [axes]

    fig.suptitle(
        dataset_labels[d],
        fontsize=16, fontweight='bold', y=1.01
    )

    for ax, img_file in zip(axes, picks):
        img = Image.open(img_file)
        ax.imshow(img)
        ax.set_xticks([])
        ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_visible(False)

    fig.tight_layout()
    save(fig, output_names[d])

# Download links
print('\nDownload links:')
for f in sorted(os.listdir(OUT)):
    if f.endswith('.pdf'):
        display(FileLink(f'{OUT}/{f}'))

In [ ]:
# Publication-Quality Figure Generation — Figures 4, 5, 6, 7
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import os
import glob
import shutil
import pandas as pd
from PIL import Image
from IPython.display import FileLink, display

# ── Global style — large fonts for journal publication ──────────────────────
plt.rcParams.update({
    'font.size': 16,
    'axes.titlesize': 18,
    'axes.labelsize': 17,
    'xtick.labelsize': 15,
    'ytick.labelsize': 15,
    'legend.fontsize': 14,
    'lines.linewidth': 2.5,
    'axes.linewidth': 1.5,
    'font.family': 'DejaVu Sans',
})

OUT = '/kaggle/output/paper_figures_large'
os.makedirs(OUT, exist_ok=True)

def save(fig, name):
    for ext in ('pdf', 'png'):
        fig.savefig(f'{OUT}/{name}.{ext}', dpi=300,
                    bbox_inches='tight', facecolor='white')
    # Copy to working for FileLink download
    shutil.copy(f'{OUT}/{name}.png', f'/kaggle/working/{name}.png')
    shutil.copy(f'{OUT}/{name}.pdf', f'/kaggle/working/{name}.pdf')
    print(f'Saved {name}')
    plt.close(fig)

SOURCE_DIR   = '/kaggle/working/source_data'
YOLO_RESULTS = '/kaggle/working/yolo_results/soil_moisture_yolo'
INFERENCE_DIR = '/kaggle/working/inference_phase6B'

# ══════════════════════════════════════════════════════════════════════════════
# FIG 4 — Laser Pattern Visual Comparison
# ══════════════════════════════════════════════════════════════════════════════
print("Generating Figure 4...")

panel_info = [
    ('soil_moisture_stir_september', 'IR Laser — Uncontrolled Field',
     'Dim/invisible laser spot\nPhase 6 accuracy: 20%', 'red'),
    ('soil_moisture_september',      'UV Laser — Field',
     'Consistent blue UV glow\nPhase 6 accuracy: 57%', 'orange'),
    ('soil-moisture-v4-uv',          'UV Laser — Controlled Lab',
     'Strong, bright UV spot\nPhase 6 accuracy: 100%', '#2E7D5E'),
    ('soil-moisture-ir',             'IR Laser — Controlled Lab',
     'Clear white IR spot\nPhase 6 accuracy: 100%', '#2E7D5E'),
]

fig, axes = plt.subplots(1, 4, figsize=(24, 8))
fig.suptitle(
    'Laser-Pattern Visual Comparison: Capture Environment Governs Performance, Not Wavelength',
    fontsize=20, fontweight='bold', y=1.02
)

for ax, (ds, modality, caption, border_col) in zip(axes, panel_info):
    img_path = None
    for split in ['test', 'valid', 'train']:
        for ext in ['*.jpg', '*.png']:
            pattern = os.path.join(SOURCE_DIR, ds, split, 'images', ext)
            matches = glob.glob(pattern)
            if matches:
                img_path = matches[0]
                break
        if img_path:
            break

    if img_path:
        img = Image.open(img_path)
        ax.imshow(img)
    else:
        ax.set_facecolor('#222222')
        ax.text(0.5, 0.5, f'[{ds}]\nimage not found',
                ha='center', va='center', color='white',
                fontsize=12, transform=ax.transAxes)

    ax.set_title(f'{ds}\n{modality}', fontsize=14,
                 fontweight='bold', color=border_col, pad=8)
    ax.set_xlabel(caption, fontsize=13, color=border_col, labelpad=6)
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_edgecolor(border_col)
        spine.set_linewidth(4)

fig.text(0.5, -0.04,
         'Left to right: IR uncontrolled field (20%), UV field (57%), '
         'UV controlled lab (100%), IR controlled lab (100%)',
         ha='center', fontsize=14, style='italic')
fig.tight_layout()
save(fig, 'fig4_laser_pattern_comparison')

# ══════════════════════════════════════════════════════════════════════════════
# FIG 5 — Phase 6 YOLOv8 Training Curves
# ══════════════════════════════════════════════════════════════════════════════
print("Generating Figure 5...")

results_path = os.path.join(YOLO_RESULTS, 'results.csv')

if not os.path.exists(results_path):
    print(f"WARNING: results.csv not found at {results_path} — skipping Fig 5")
else:
    df = pd.read_csv(results_path)
    df.columns = df.columns.str.strip()
    print(f"Columns: {list(df.columns)}")

    best_ep = 32

    fig, axes = plt.subplots(2, 2, figsize=(18, 14))
    fig.suptitle(
        'Phase 6 YOLOv8 Training Curves — Best Epoch 32/42 (EarlyStopping patience=10)',
        fontsize=20, fontweight='bold'
    )

    def plot_panel(ax, col_train, col_val, title, ylabel, best_epoch):
        cols = df.columns.tolist()
        train_col = next((c for c in cols if col_train.lower() in c.lower()), None)
        val_col   = next((c for c in cols if col_val.lower() in c.lower()), None)
        if train_col:
            ax.plot(df[train_col], label='Train',
                    color='#4C72B0', linewidth=2.5)
        if val_col:
            ax.plot(df[val_col], label='Val',
                    color='#C0392B', linewidth=2.5)
        ax.axvline(best_epoch, color='#888888', linestyle='--',
                   linewidth=2.0, label=f'Best epoch {best_epoch}')
        if title == 'mAP50' and val_col:
            best_val = df[val_col].max()
            ax.annotate(
                f'Best: {best_val:.3f} ({best_val*100:.1f}%)',
                xy=(best_epoch, best_val),
                xytext=(best_epoch + 2, best_val - 0.05),
                fontsize=13, color='#333333',
                arrowprops=dict(arrowstyle='->', lw=1.5)
            )
        ax.set_title(title, fontsize=17, fontweight='bold')
        ax.set_xlabel('Epoch', fontsize=15)
        ax.set_ylabel(ylabel, fontsize=15)
        ax.legend(fontsize=13)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.yaxis.grid(True, linestyle='--', alpha=0.4)

    plot_panel(axes[0,0], 'box_loss',          'val/box',
               'Box Loss',            'Loss',   best_ep)
    plot_panel(axes[0,1], 'cls_loss',          'val/cls',
               'Classification Loss', 'Loss',   best_ep)
    plot_panel(axes[1,0], 'metrics/mAP50',     'metrics/mAP50',
               'mAP50',               'mAP50',  best_ep)
    plot_panel(axes[1,1], 'metrics/precision', 'metrics/recall',
               'Precision and Recall','Value',  best_ep)

    fig.tight_layout()
    save(fig, 'fig5_phase6_training_curves')

# ══════════════════════════════════════════════════════════════════════════════
# FIG 6 — Phase 6 Normalised Confusion Matrix
# ══════════════════════════════════════════════════════════════════════════════
print("Generating Figure 6...")

cm_path = os.path.join(YOLO_RESULTS, 'confusion_matrix_normalized.png')

if not os.path.exists(cm_path):
    print(f"WARNING: confusion_matrix_normalized.png not found — skipping Fig 6")
else:
    cm_img = Image.open(cm_path)
    fig, ax = plt.subplots(figsize=(14, 14))
    ax.imshow(cm_img)
    ax.axis('off')
    ax.set_title(
        'Phase 6 Normalised Confusion Matrix — YOLOv8 (95.3% mAP50)',
        fontsize=18, fontweight='bold', pad=15
    )
    fig.tight_layout()
    save(fig, 'fig6_confusion_matrix')

# ══════════════════════════════════════════════════════════════════════════════
# FIG 7 — Phase 6 Inference Grid
# ══════════════════════════════════════════════════════════════════════════════
print("Generating Figure 7...")

if not os.path.exists(INFERENCE_DIR):
    print(f"WARNING: {INFERENCE_DIR} not found — skipping Fig 7. Run Step 27 first.")
else:
    img_files = sorted(
        glob.glob(f'{INFERENCE_DIR}/*.jpg') +
        glob.glob(f'{INFERENCE_DIR}/*.png')
    )[:46]

    if not img_files:
        print(f"No images found in {INFERENCE_DIR}")
    else:
        print(f"Found {len(img_files)} inference images")
        ncols = 6
        nrows = int(np.ceil(len(img_files) / ncols))

        fig, axes = plt.subplots(nrows, ncols,
                                 figsize=(ncols * 4, nrows * 4))
        fig.suptitle(
            'Phase 6 Inference Results — 41/46 Correct (89.1%) '
            'Across All Seven Datasets',
            fontsize=20, fontweight='bold', y=1.01
        )

        dataset_colors = {
            'soil-moisture-v4':             '#185FA5',
            'soil-moisture-v4-ir':          '#1D9E75',
            'soil-moisture-v4-uv':          '#533AB7',
            'soil-moisture-ir':             '#BA7517',
            'soil-moisture-5sagf':          '#0F6E56',
            'soil_moisture_september':      '#EF9F27',
            'soil_moisture_stir_september': '#E24B4A',
        }

        for idx, (ax, img_file) in enumerate(
                zip(axes.flatten(), img_files)):
            img      = Image.open(img_file)
            img_name = os.path.basename(img_file)
            ax.imshow(img)
            ax.axis('off')

            # Detect correct/incorrect from bottom strip color
            img_arr    = np.array(img)
            bottom_row = img_arr[-10:, :, :]
            mean_r     = bottom_row[:, :, 0].mean()
            mean_g     = bottom_row[:, :, 1].mean()
            is_correct = mean_g > mean_r

            # Tick or cross overlay
            marker       = '✓' if is_correct else '✗'
            marker_color = '#1D9E75' if is_correct else '#E24B4A'
            ax.text(0.95, 0.95, marker,
                    transform=ax.transAxes,
                    fontsize=16, color=marker_color,
                    ha='right', va='top', fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.2',
                              facecolor='white', alpha=0.8))

            # Dataset border color
            border_col = '#888888'
            for ds, col in dataset_colors.items():
                if ds in img_name:
                    border_col = col
                    break
            for spine in ax.spines.values():
                spine.set_edgecolor(border_col)
                spine.set_linewidth(2)

        # Hide unused axes
        for ax in axes.flatten()[len(img_files):]:
            ax.axis('off')

        fig.tight_layout()
        save(fig, 'fig7_inference_grid')

# ── Summary and download links ───────────────────────────────────────────────
print(f"\nAll figures saved to {OUT}")
print("Download links:")
for f in sorted(os.listdir(OUT)):
    if f.endswith('.pdf'):
        display(FileLink(f'/kaggle/working/{f}'))

In [ ]:
import os

# Check working directory
working_figures = '/kaggle/working/paper_figures'
if os.path.exists(working_figures):
    print("Files in /kaggle/working/paper_figures/:")
    for f in os.listdir(working_figures):
        print(f"  {f}")
else:
    print("paper_figures folder not found in working directory")

# Check output directory
output_figures = '/kaggle/output/paper_figures'
if os.path.exists(output_figures):
    print("\nFiles in /kaggle/output/paper_figures/:")
    for f in os.listdir(output_figures):
        print(f"  {f}")
else:
    print("paper_figures folder not found in output directory")

# Check all contents of output
print("\nAll contents of /kaggle/output/:")
if os.path.exists('/kaggle/output'):
    for f in os.listdir('/kaggle/output'):
        print(f"  {f}")
else:
    print("output directory not found")